<a href="https://colab.research.google.com/github/technologyhamed/LLMAttack/blob/main/step2_apt_sequence_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# پروژهٔ پیش‌بینی تکنیک بعدی ATT&CK — نوت‌بوک تجمیعی (گام ۱ + گام ۲)

# گام ۱ — ساخت دیتاست `Apt-sequence.json` از فایل MITRE ATT&CK

**هدف:** تولید دیتاست پیش‌بینی تکنیک بعدی (`next_technique_prediction`) با واحد دنباله = رتبهٔ تاکتیک (`tactic_rank`)، دقیقاً مطابق قالب نمونهٔ ارسالی.

**منبع:** `enterprise-attack-v19_2.xlsx` (شیت‌های `techniques`, `tactics`, `groups`, `campaigns`, `relationships`, `software`)

## منطق ساخت نمونه‌ها (مطابق قالب نمونه)
1. برای هر «کمپین» (گروه APT یا کمپین ATT&CK) تکنیک‌های استفاده‌شده جمع می‌شود و هر تکنیک به تاکتیک/رتبهٔ آن نگاشت می‌شود.
2. تکنیک‌ها بر اساس رتبهٔ تاکتیک (ترتیب Kill-Chain) گروه‌بندی می‌شوند.
3. برای هر جفت رتبهٔ متوالی `(r_i → r_{i+1})`: ورودی = **همهٔ** تکنیک‌های رتبه‌های ≤ `r_i` (به‌ترتیب رتبه)، و به ازای **هر** تکنیک رتبهٔ `r_{i+1}` یک نمونه با آن `target` ساخته می‌شود.
4. **پنجرهٔ لغزان (اختیاری):** با `WINDOW_SIZE = N` فقط N تکنیک آخر تاریخچه به‌عنوان ورودی نگه داشته می‌شود. مقدار پیش‌فرض `None` است (تاریخچهٔ کامل، مطابق نمونهٔ ارسالی).

## تصمیم‌های طراحی
- **رتبه‌ها:** `reconnaissance=0 … impact=14` بر اساس ترتیب ماتریس نسخهٔ v19 (در این نسخه `defense-evasion` به `stealth` و `defense-impairment` تقسیم شده است؛ پس رتبه‌ها با نمونهٔ قدیمی کمی فرق دارد، مثلاً C2 = 12 به‌جای 11).
- **تکنیک چندتاکتیکی:** پیش‌فرض `first` (اولین تاکتیک در ترتیب Kill-Chain). حالت `expand` هم پیاده شده است.
- **فیلدهای `object` و `target_area`:** در فایل اکسل ATT&CK فیلد مستقل ندارند؛ با کلیدواژه از توضیحات گروه/کمپین استخراج می‌شوند (heuristic) و در صورت نبود اطلاعات `Unknown` می‌شوند.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json, re, collections
from pathlib import Path
import pandas as pd

# ----------------------------- تنظیمات -----------------------------
XLSX_CANDIDATES = [
    "/content/enterprise-attack-v19_2.xlsx",
    "enterprise-attack-v19_2.xlsx",
]
XLSX_PATH = next((p for p in XLSX_CANDIDATES if Path(p).exists()), None)
assert XLSX_PATH, "فایل اکسل پیدا نشد؛ مسیر را در XLSX_CANDIDATES اصلاح کنید."

OUTPUT_DIR  = Path("/content/user-data/outputs") if Path("/content/user-data/outputs").exists() else Path(".")
OUTPUT_PATH = OUTPUT_DIR / "Apt-sequence.json"

INCLUDE_GROUPS              = True    # گروه‌های APT (مثل APT-C-36)
INCLUDE_ATTACK_CAMPAIGNS    = True    # کمپین‌های رسمی ATT&CK (C00xx)
INCLUDE_SOFTWARE_TECHNIQUES = True    # افزودن تکنیک‌های نرم‌افزارهای مورداستفادهٔ گروه
MULTI_TACTIC_POLICY         = "first" # "first" | "expand"
WINDOW_SIZE                 = None    # None = تاریخچهٔ کامل | عدد N = پنجرهٔ لغزان
MIN_DISTINCT_RANKS          = 2       # حداقل تعداد رتبهٔ متمایز برای معتبر بودن کمپین

print("source:", XLSX_PATH)
print("output:", OUTPUT_PATH)

In [ ]:
# ----------------------------- بارگذاری شیت‌ها -----------------------------
xl = pd.ExcelFile(XLSX_PATH)
techniques = xl.parse("techniques")
tactics    = xl.parse("tactics")
groups     = xl.parse("groups")
campaigns  = xl.parse("campaigns")
rels       = xl.parse("relationships")

print({k: v.shape for k, v in dict(techniques=techniques, tactics=tactics, groups=groups,
                                    campaigns=campaigns, relationships=rels).items()})

In [ ]:
# ----------------------------- ترتیب و رتبهٔ تاکتیک‌ها -----------------------------
def slug(name: str) -> str:
    return re.sub(r"\s+", "-", str(name).strip().lower())

# ترتیب ستون‌های ماتریس Enterprise ATT&CK v19
TACTIC_ORDER = [
    "reconnaissance", "resource-development", "initial-access", "execution",
    "persistence", "privilege-escalation", "stealth", "defense-impairment",
    "credential-access", "discovery", "lateral-movement", "collection",
    "command-and-control", "exfiltration", "impact",
]
TACTIC_RANK = {t: i for i, t in enumerate(TACTIC_ORDER)}

# اعتبارسنجی با شیت tactics (اگر نسخهٔ ATT&CK عوض شود اینجا خطا می‌دهد)
sheet_tactics = {slug(n) for n in tactics["name"]}
assert sheet_tactics == set(TACTIC_ORDER), (sheet_tactics ^ set(TACTIC_ORDER))
print(TACTIC_RANK)

In [ ]:
# ----------------------------- نگاشت تکنیک -> [(rank, tactic)] -----------------------------
def build_technique_tactics(df: pd.DataFrame, policy: str = "first") -> dict:
    mapping = {}
    for tid, tac in zip(df["ID"], df["tactics"]):
        if pd.isna(tac):
            continue
        pairs = sorted({(TACTIC_RANK[slug(t)], slug(t)) for t in str(tac).split(",") if slug(t) in TACTIC_RANK})
        if not pairs:
            continue
        mapping[tid] = pairs[:1] if policy == "first" else pairs
    return mapping

TECH_TACTICS = build_technique_tactics(techniques, MULTI_TACTIC_POLICY)
print("techniques with tactic:", len(TECH_TACTICS), "/", len(techniques))
print("T1134 ->", TECH_TACTICS["T1134"])

In [ ]:
# ----------------------------- استخراج object / target_area از توضیحات (heuristic) -----------------------------
def clean_text(s) -> str:
    if pd.isna(s):
        return ""
    s = re.sub(r"\(Citation:[^)]*\)", " ", str(s))
    s = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", s)   # [text](url) -> text
    return re.sub(r"\s+", " ", s).strip()

OBJECT_PATTERNS = {
    "Cyber Espionage": r"espionage|intelligence (collection|gathering)|surveillance|\bspy",
    "Financial":       r"financially[- ]motivated|financial gain|financially driven|monetary|ransomware|extortion|cybercrim|criminal|\bfraud|\bprofit",
    "Destruction":     r"destructive|wiper|sabotage|disruptive|disrupt",
    "Hacktivism":      r"hacktivis",
}
REGION_PATTERNS = {
    "Latin America":   r"latin america|south america|central america|colombia|brazil|mexico|argentina|chile|peru|ecuador|venezuela",
    "North America":   r"united states|u\.s\.|north america|canada",
    "Europe":          r"europe|ukrain|poland|germany|france|united kingdom|\buk\b|italy|spain|nato|baltic|nordic",
    "Russia and CIS":  r"russia|belarus|kazakhstan|\bcis\b",
    "Middle East":     r"middle east|israel|iran|saudi|\buae\b|united arab emirates|turkey|iraq|syria|lebanon|palestin|qatar|kuwait",
    "East Asia":       r"china|taiwan|japan|korea|hong kong|mongolia",
    "South Asia":      r"south asia|india|pakistan|bangladesh|sri lanka|nepal",
    "Southeast Asia":  r"southeast asia|vietnam|thailand|indonesia|malaysia|philippines|singapore|myanmar|cambodia|\blaos\b",
    "Africa":          r"africa|nigeria|kenya|egypt|ethiopia|morocco|algeria|tunisia|libya|sudan|ghana",
    "Global":          r"worldwide|global(ly)?\b|across the globe|multiple regions|various countries",
}
SECTOR_PATTERNS = {
    "Government":              r"government|ministr|diplomat|embass|municipal|public sector",
    "Defense":                 r"defen[cs]e|military|armed forces|aerospace",
    "Financial":               r"financial (sector|institution|service|entit|industr|organi)|\bin the financial\b|\bbanks?\b|banking|cryptocurrenc|fintech|insurance",
    "Energy":                  r"energy|electric|power (grid|plant|sector)|\boil\b|\bgas\b|utilit|nuclear",
    "Telecommunications":      r"telecom",
    "Healthcare":              r"health|pharma|medical|hospital|biotech",
    "Education":               r"educat|universit|academi|research (institution|organization)",
    "Manufacturing":           r"manufactur|industrial|automotive|semiconductor",
    "Technology":              r"technology|software|managed service|cloud|supply chain",
    "Media":                   r"(?<!social )media\b|journalis",
    "Critical Infrastructure": r"critical infrastructure|\bics\b|scada|water (utilit|sector|treatment)|transportation|logistic|shipping|aviation|maritime",
    "Retail and Hospitality":  r"retail|hospitality|restaurant|e-?commerce|casino|gaming",
    "NGO and Civil Society":   r"\bngo|non-?profit|human rights|dissident|activist|think tank|civil society",
    "Legal":                   r"law firm|legal",
}
TARGET_SENT = re.compile(r"\btarget(ed|s|ing)?\b|victim|compromis", re.I)

def _match(patterns: dict, text: str) -> list:
    return [k for k, p in patterns.items() if re.search(p, text, re.I)]

def infer_object(desc: str) -> str:
    found = _match(OBJECT_PATTERNS, desc)
    if not found:
        return "Unknown"
    return found[0] if len(found) == 1 else ", ".join(found[:-1]) + " and " + found[-1]

def infer_target_area(desc: str) -> str:
    sents = [s for s in re.split(r"(?<=[.!?])\s+", desc) if TARGET_SENT.search(s)]
    text = " ".join(sents)
    found = _match(REGION_PATTERNS, text) + _match(SECTOR_PATTERNS, text)
    return ", ".join(found) if found else "Unknown"

d = clean_text(groups.loc[groups["name"] == "APT-C-36", "description"].iloc[0])
print(infer_object(d), "|", infer_target_area(d))

In [ ]:
# ----------------------------- ساخت موجودیت‌ها (گروه‌ها / کمپین‌ها) -----------------------------
uses = rels[rels["mapping type"] == "uses"]

grp_tech  = uses[(uses["source type"] == "group")    & (uses["target type"] == "technique")].groupby("source ID")["target ID"].apply(set).to_dict()
camp_tech = uses[(uses["source type"] == "campaign") & (uses["target type"] == "technique")].groupby("source ID")["target ID"].apply(set).to_dict()
grp_sw    = uses[(uses["source type"] == "group")    & (uses["target type"] == "software")].groupby("source ID")["target ID"].apply(set).to_dict()
sw_tech   = uses[(uses["source type"] == "software") & (uses["target type"] == "technique")].groupby("source ID")["target ID"].apply(set).to_dict()

def make_entities() -> list:
    ents = []
    if INCLUDE_GROUPS:
        for _, row in groups.iterrows():
            techs = set(grp_tech.get(row["ID"], set()))
            if INCLUDE_SOFTWARE_TECHNIQUES:
                for s in grp_sw.get(row["ID"], set()):
                    techs |= sw_tech.get(s, set())
            ents.append(dict(id=row["ID"], name=row["name"], kind="group", techniques=techs, description=clean_text(row["description"])))
    if INCLUDE_ATTACK_CAMPAIGNS:
        for _, row in campaigns.iterrows():
            ents.append(dict(id=row["ID"], name=row["name"], kind="campaign", techniques=set(camp_tech.get(row["ID"], set())), description=clean_text(row["description"])))
    # یکتا کردن نام‌ها
    cnt = collections.Counter(e["name"] for e in ents)
    for e in ents:
        if cnt[e["name"]] > 1:
            e["name"] = f'{e["name"]} ({e["id"]})'
    return ents

ENTITIES = make_entities()
print(len(ENTITIES), collections.Counter(e["kind"] for e in ENTITIES))

In [ ]:
# ----------------------------- CampaignSequenceBuilder -----------------------------
class CampaignSequenceBuilder:
    """ساخت نمونه‌های next-technique با واحد tactic_rank (ورودی تجمعی یا پنجرهٔ لغزان)."""
    NAME, VERSION = "CampaignSequenceBuilder", "1.0"

    def __init__(self, technique_tactics, window_size=None, min_distinct_ranks=2):
        self.tt = technique_tactics
        self.window = window_size
        self.min_ranks = min_distinct_ranks
        self.unknown_techniques = collections.Counter()

    def _items_by_rank(self, techs):
        by_rank = collections.defaultdict(list)
        for t in sorted(techs):
            if t not in self.tt:
                self.unknown_techniques[t] += 1
                continue
            for rank, _ in self.tt[t]:
                by_rank[rank].append(t)
        return by_rank

    def build_campaign(self, ent):
        by_rank = self._items_by_rank(ent["techniques"])
        ranks = sorted(by_rank)
        if len(ranks) < self.min_ranks:
            return None                      # کمپین نامعتبر
        obj  = infer_object(ent["description"])
        area = infer_target_area(ent["description"])
        history, seen, samples = [], set(), []
        for i, rk in enumerate(ranks):
            if i > 0:
                prev = ranks[i - 1]
                inp = history[-self.window:] if self.window else list(history)
                for tgt in by_rank[rk]:
                    if tgt in seen:          # تکنیکی که قبلاً در تاریخچه بوده هدف نمی‌شود
                        continue
                    samples.append({
                        "campaign": ent["name"],
                        "input": list(inp),
                        "target": tgt,
                        "input_tactic_rank": prev,
                        "target_tactic_rank": rk,
                        "input_tactic": TACTIC_ORDER[prev],
                        "target_tactic": TACTIC_ORDER[rk],
                        "object": obj,
                        "target_area": area,
                    })
            for t in by_rank[rk]:
                if t not in seen:
                    seen.add(t); history.append(t)
        return samples or None

    def build(self, entities, source_name):
        samples, valid, skipped = [], 0, 0
        for ent in entities:
            res = self.build_campaign(ent)
            if res is None:
                skipped += 1
            else:
                valid += 1; samples.extend(res)
        unique = len({(tuple(s["input"]), s["target"]) for s in samples})
        meta = {
            "builder": self.NAME, "version": self.VERSION,
            "source": source_name,
            "task": "next_technique_prediction",
            "sequence_unit": "tactic_rank",
            "total_campaigns": len(entities),
            "valid_campaigns": valid,
            "skipped_campaigns": skipped,
            "total_samples": len(samples),
            "unique_samples": unique,
            "config": {
                "window_size": self.window, "multi_tactic_policy": MULTI_TACTIC_POLICY,
                "include_groups": INCLUDE_GROUPS, "include_attack_campaigns": INCLUDE_ATTACK_CAMPAIGNS,
                "include_software_techniques": INCLUDE_SOFTWARE_TECHNIQUES,
                "min_distinct_ranks": self.min_ranks, "tactic_rank": TACTIC_RANK,
            },
        }
        return {"metadata": meta, "samples": samples}

builder = CampaignSequenceBuilder(TECH_TACTICS, window_size=WINDOW_SIZE, min_distinct_ranks=MIN_DISTINCT_RANKS)
dataset = builder.build(ENTITIES, Path(XLSX_PATH).name)
print(json.dumps(dataset["metadata"], indent=2, ensure_ascii=False)[:900])
print("unknown techniques (not in sheet/tactic):", len(builder.unknown_techniques))

In [ ]:
# ----------------------------- ذخیرهٔ خروجی -----------------------------
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)
print("saved:", OUTPUT_PATH, f"({OUTPUT_PATH.stat().st_size/1e6:.2f} MB)")

In [ ]:
# ----------------------------- تست‌های اعتبارسنجی -----------------------------
with open(OUTPUT_PATH, encoding="utf-8") as f:
    ds = json.load(f)
meta, S = ds["metadata"], ds["samples"]

KEYS = ["campaign","input","target","input_tactic_rank","target_tactic_rank","input_tactic","target_tactic","object","target_area"]
assert meta["total_samples"] == len(S) > 0
assert meta["valid_campaigns"] + meta["skipped_campaigns"] == meta["total_campaigns"]
assert meta["unique_samples"] <= meta["total_samples"]
for s in S:
    assert list(s.keys()) == KEYS
    assert s["input"] and s["target"] not in s["input"]
    assert s["input_tactic_rank"] < s["target_tactic_rank"]
    assert TACTIC_ORDER[s["input_tactic_rank"]] == s["input_tactic"]
    assert TACTIC_ORDER[s["target_tactic_rank"]] == s["target_tactic"]
    assert len(s["input"]) == len(set(s["input"]))
    if WINDOW_SIZE: assert len(s["input"]) <= WINDOW_SIZE
    if MULTI_TACTIC_POLICY == "first":
        assert TECH_TACTICS[s["target"]][0][0] == s["target_tactic_rank"]
        assert TECH_TACTICS[s["input"][-1]][0][0] == s["input_tactic_rank"]   # آخرین ورودی در رتبهٔ ورودی است
        ranks_in = [TECH_TACTICS[t][0][0] for t in s["input"]]
        assert ranks_in == sorted(ranks_in)                                    # ورودی به‌ترتیب رتبه
print("✓ all structural checks passed on", len(S), "samples")

# بررسی نمونه‌ای مشابه قالب ارسالی
ex = next(s for s in S if s["campaign"] == "APT-C-36")
print(json.dumps(ex, indent=2, ensure_ascii=False))

In [ ]:
# ----------------------------- آمار توصیفی -----------------------------
df = pd.DataFrame(S)
print("campaigns with samples:", df["campaign"].nunique())
print("\nsamples per target tactic:")
print(df["target_tactic"].value_counts().reindex(TACTIC_ORDER).dropna().astype(int).to_string())
print("\ninput length (min/median/max):", df["input"].map(len).min(), df["input"].map(len).median(), df["input"].map(len).max())
print("\nobject == Unknown     :", f'{(df["object"]=="Unknown").mean():.1%}')
print("target_area == Unknown:", f'{(df["target_area"]=="Unknown").mean():.1%}')
print("\ntop-5 campaigns by #samples:")
print(df["campaign"].value_counts().head(5).to_string())

---
# گام ۲ — مدل‌های HMM / SHHMM / VOHMM برای پیش‌بینی تکنیک بعدی

سلول‌های بالا (گام ۱) بدون تغییر منطق آمده‌اند؛ فقط `INCLUDE_SOFTWARE_TECHNIQUES=True` شده تا ۲۲۹ کمپین معتبر به دست آید.
از اینجا به بعد گام ۲ است.

| مورد | تصمیم |
|---|---|
| نوع یادگیری | **نظارتی** برای هر سه مدل (تابع هزینهٔ اصلی: NLL تکنیک بعدی) |
| پنجرهٔ تاریخچه | `WINDOW=16` تکنیک آخر (خالی گذاشتن = تاریخچهٔ کامل تا ۲۳۰ گام، که هزینه را بالا می‌برد و سیگنال مفیدی اضافه نمی‌کند) |
| تقسیم | ۱۰٪ گروه‌ها = FINAL test (یک بار مصرف)، ۹۰٪ = Development با ۱۰-fold group-disjoint |
| گروه | کمپین‌های منتسب به یک گروه + کمپین‌های با Jaccard ≥ 0.9 در یک «گروه» ادغام می‌شوند تا نشت نداشته باشیم |
| بدون validation | تعداد epoch ثابت، LR و α فقط با CV روی Development انتخاب می‌شود |
| ذخیره | فقط بهترین مدل (به‌همراه گزارش یکپارچگی reload) |

In [ ]:
# ============================== تنظیمات گام ۲ ==============================
import os, sys, json, math, time, copy, hashlib, pickle, random, collections, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import rankdata
import matplotlib.pyplot as plt

try:
    import torch, torch.nn as nn, torch.nn.functional as F
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

CFG = dict(
    SEED=42, WINDOW=16, N_FOLDS=10, FINAL_FRAC=0.10, JACCARD_MERGE=0.90,
    FAST_MODE=False,                                   # True = اجرای دودی روی GPU (LR و epoch کم)
    # --- HMM ---
    HMM_LRS=[3e-3, 1e-2, 3e-2], HMM_EPOCHS=40, HMM_BS=512, LAM_STATE=0.05,
    A_SMOOTH=0.02, B_SMOOTH=1e-3, PI_SMOOTH=0.5,
    # --- SHHMM ---
    SH_LRS=[3e-4, 1e-3, 3e-3], SH_EPOCHS=40, SH_BS=256, SH_WD=1e-2,
    SH_EMB=128, SH_HID=256, SH_LAYERS=2, SH_DROP=0.2,
    LAM_AUX=0.3, LAM_HMM=0.3, LABEL_SMOOTH=0.05,
    AUG=dict(p_view=0.8, p_shuffle=0.5, p_trunc=0.3, p_drop=0.10),
    # --- نامتوازنی ---
    IMBALANCE="effective_number",                      # none | effective_number | logit_adjust
    EFF_BETA=0.99, LOGIT_TAU=1.0,
    # --- VOHMM ---
    VOM_D=[0.6, 0.8, 0.95], VOM_EPS=1e-3,
    ALPHAS=[0.0, 0.25, 0.5, 0.75, 1.0],
    # --- ارزیابی ---
    SELECT_METRIC="mrr_open", BOOT=1000, PERM=1000, CM_TOP=20,
    MASK_SEEN=True,      # تکنیک موجود در ورودی هرگز target نمی‌شود (قید دیتاست) — برای هر سه مدل یکسان
)
if CFG["FAST_MODE"]:
    CFG.update(HMM_LRS=[1e-2], SH_LRS=[1e-3], HMM_EPOCHS=3, SH_EPOCHS=3, N_FOLDS=3)

NA = -1.0                                              # مقدار «نامربوط» در کلید کانفیگ‌ها
OUT2 = Path(os.environ.get("STEP2_OUT", str(OUTPUT_DIR / "step2")))
FIG_DIR = OUT2 / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

if HAS_TORCH:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    USE_AMP = DEVICE.type == "cuda"
    if DEVICE.type == "cuda":
        torch.backends.cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
else:
    DEVICE, USE_AMP = None, False

def seed_all(s):
    random.seed(s); np.random.seed(s)
    if HAS_TORCH:
        torch.manual_seed(s)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(s)

seed_all(CFG["SEED"])
print("torch:", torch.__version__ if HAS_TORCH else "NOT INSTALLED (SHHMM غیرفعال، HMM در حالت MLE)",
      "| device:", DEVICE, "| AMP(bf16):", USE_AMP)
print("output dir:", OUT2)

In [ ]:
# ============================== بارگذاری دیتاست و آرایه‌های سراسری ==============================
with open(OUTPUT_PATH, encoding="utf-8") as f:
    DS = json.load(f)
SAMPLES = DS["samples"]; N_S = len(SAMPLES)
L = CFG["WINDOW"]; NT = len(TACTIC_ORDER)
assert DS["metadata"]["total_samples"] == N_S

PRIMARY = {t: v[0][0] for t, v in TECH_TACTICS.items()}            # تاکتیک اصلی هر تکنیک
ALL_IDS = sorted({t for s in SAMPLES for t in s["input"]} | {s["target"] for s in SAMPLES},
                 key=lambda t: (PRIMARY[t], t))                      # gid صعودی = ترتیب کانونی زنجیره
GID = {t: i for i, t in enumerate(ALL_IDS)}
TACT_OF_GID = np.array([PRIMARY[t] for t in ALL_IDS], dtype=np.int64)
TECH_NAME = dict(zip(techniques["ID"], techniques["name"]))

camp_list, CAMP_IDX = np.unique(np.array([s["campaign"] for s in SAMPLES]), return_inverse=True)
GX = np.full((N_S, L), -1, np.int32)                                 # پنجرهٔ ورودی، راست‌پد
GLEN = np.zeros(N_S, np.int32); ORIG_LEN = np.zeros(N_S, np.int32)
GY = np.zeros(N_S, np.int32); GYT = np.zeros(N_S, np.int32)
chain_sets = [set() for _ in camp_list]
for i, s in enumerate(SAMPLES):
    ids = [GID[t] for t in s["input"]]
    w = ids[-L:]
    GX[i, :len(w)] = w; GLEN[i] = len(w); ORIG_LEN[i] = len(ids)
    GY[i] = GID[s["target"]]; GYT[i] = s["target_tactic_rank"]
    chain_sets[CAMP_IDX[i]].update(ids); chain_sets[CAMP_IDX[i]].add(int(GY[i]))
CHAIN = [np.array(sorted(c), np.int64) for c in chain_sets]          # زنجیرهٔ کانونی هر کمپین
GTRUNC = ORIG_LEN > L
if MULTI_TACTIC_POLICY == "first":
    assert (TACT_OF_GID[GY] == GYT).all()
assert not any(GY[i] in set(GX[i, :GLEN[i]]) for i in range(0, N_S, 97)), "target داخل ورودی است"
print(f"samples={N_S}  techniques={len(ALL_IDS)}  campaigns={len(camp_list)}  window L={L}  truncated={GTRUNC.mean():.1%}")

In [ ]:
# ============================== گروه‌بندی بدون نشت (ادغام کمپین/گروه منتسب و تکراری‌ها) ==============================
ent_by_id = {e["id"]: e for e in ENTITIES}
parent = {c: c for c in camp_list}
def _find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
def _union(a, b):
    ra, rb = _find(a), _find(b)
    if ra != rb:
        parent[max(ra, rb)] = min(ra, rb)

n_attr = 0
for _, r in rels[rels["mapping type"] == "attributed-to"].iterrows():
    a, b = ent_by_id.get(r["source ID"]), ent_by_id.get(r["target ID"])
    if a and b and a["name"] in parent and b["name"] in parent:
        _union(a["name"], b["name"]); n_attr += 1

n_jac = 0
for i in range(len(camp_list)):
    for j in range(i + 1, len(camp_list)):
        inter = len(chain_sets[i] & chain_sets[j])
        if inter and inter / (len(chain_sets[i]) + len(chain_sets[j]) - inter) >= CFG["JACCARD_MERGE"]:
            if _find(camp_list[i]) != _find(camp_list[j]):
                n_jac += 1
            _union(camp_list[i], camp_list[j])

GROUP_NAMES, GROUP_IDX = np.unique(np.array([_find(c) for c in camp_list])[CAMP_IDX], return_inverse=True)
G_SIZES = np.bincount(GROUP_IDX)
print(f"campaigns={len(camp_list)} -> independent groups={len(GROUP_NAMES)}  (attributed-to merges={n_attr}, jaccard merges={n_jac})")

In [ ]:
# ============================== تقسیم: FINAL test + 10-fold GroupKFold روی Development ==============================
def balanced_group_kfold(group_ids, sizes, k, seed):
    rng = np.random.default_rng(seed)
    gs = list(rng.permutation(group_ids))
    gs.sort(key=lambda g: -sizes[g])                   # LPT: بزرگ‌ترین‌ها اول، tie تصادفی (sort پایدار)
    loads = np.zeros(k); folds = [[] for _ in range(k)]
    for g in gs:
        j = int(np.argmin(loads)); folds[j].append(g); loads[j] += sizes[g]
    return folds

n_groups = len(GROUP_NAMES); n_test_g = max(1, round(CFG["FINAL_FRAC"] * n_groups))
best_try, best_gap = None, 9
for attempt in range(2000):
    pick = np.random.default_rng(CFG["SEED"] + attempt).permutation(n_groups)[:n_test_g]
    share = G_SIZES[pick].sum() / N_S
    if abs(share - 0.10) < best_gap:
        best_try, best_gap = pick, abs(share - 0.10)
    if 0.09 <= share <= 0.11:
        break
FINAL_GROUPS = np.sort(best_try)
FINAL_IDX = np.flatnonzero(np.isin(GROUP_IDX, FINAL_GROUPS))
DEV_IDX = np.flatnonzero(~np.isin(GROUP_IDX, FINAL_GROUPS))
dev_groups = np.setdiff1d(np.arange(n_groups), FINAL_GROUPS)

fold_groups = balanced_group_kfold(dev_groups, G_SIZES, CFG["N_FOLDS"], CFG["SEED"])
FOLDS = []
for f in range(CFG["N_FOLDS"]):
    te = np.flatnonzero(np.isin(GROUP_IDX, fold_groups[f]))
    tr = np.setdiff1d(DEV_IDX, te)
    FOLDS.append((tr, te))

# ---- assertهای نشت ----
assert not set(GROUP_IDX[DEV_IDX]) & set(GROUP_IDX[FINAL_IDX])
assert len(DEV_IDX) + len(FINAL_IDX) == N_S
for tr, te in FOLDS:
    assert not set(GROUP_IDX[tr]) & set(GROUP_IDX[te]), "نشت گروه بین train و test"
    assert not set(tr) & set(te)
assert sorted(np.concatenate([te for _, te in FOLDS])) == sorted(DEV_IDX), "هر نمونهٔ Dev باید دقیقاً یک بار test شود"
print(f"FINAL: {len(FINAL_GROUPS)} groups / {len(FINAL_IDX)} samples ({len(FINAL_IDX)/N_S:.1%}) | Dev: {len(dev_groups)} groups / {len(DEV_IDX)} samples")
print("fold sizes (samples):", [len(te) for _, te in FOLDS])

In [ ]:
# ============================== Vocabulary (فقط از train) و بردارهای وزن کلاس ==============================
class Vocab:
    """ورودی: 0=PAD 1=UNK 2+v ; خروجی: کلاس v در [0,n). فقط از نمونه‌های train ساخته می‌شود."""
    def __init__(self, tr_idx):
        gx = GX[tr_idx]
        self.gids = np.unique(np.concatenate([gx[gx >= 0], GY[tr_idx]])).astype(np.int64)
        self.n = len(self.gids)
        self.map = np.full(len(ALL_IDS), -1, np.int32); self.map[self.gids] = np.arange(self.n)
        self.tact = TACT_OF_GID[self.gids]                                   # تاکتیک هر کلاس
        self.names = [ALL_IDS[g] for g in self.gids]
        self.counts = np.bincount(self.map[GY[tr_idx]], minlength=self.n)    # فراوانی targetها در train
        self.tact_tok = np.concatenate([[NT, NT], self.tact]).astype(np.int64)   # تاکتیک توکن‌ها (NT=نامشخص)
    def encode(self, idx):
        gx = GX[idx]; v = self.map[np.where(gx >= 0, gx, 0)]
        tok = np.where(gx < 0, 0, np.where(v < 0, 1, v + 2)).astype(np.int64)
        return tok, GLEN[idx].astype(np.int64), GTRUNC[idx].copy()
    def target(self, idx):
        return self.map[GY[idx]].astype(np.int64)                            # -1 = خارج از vocabulary

def class_weight_vector(counts, mode, beta):
    c = counts.astype(np.float64)
    if mode == "effective_number":
        w = np.where(c > 0, (1 - beta) / (1 - np.power(beta, np.maximum(c, 1))), 0.0)
        w[c == 0] = w[c > 0].mean()
        return w * c.sum() / np.sum(w * c)                                   # میانگین وزن نمونه = 1
    return np.ones_like(c)

def log_prior(counts):
    return np.log((counts + 1.0) / (counts.sum() + len(counts)))

_v = Vocab(FOLDS[0][0])
_tok, _ln, _tr = _v.encode(FOLDS[0][1]); _y = _v.target(FOLDS[0][1])
print(f"fold0: vocab={_v.n}  test target coverage={(_y >= 0).mean():.3f}  tok shape={_tok.shape}")
assert (_y < 0).sum() == (~np.isin(GY[FOLDS[0][1]], _v.gids)).sum()          # OOV فقط از train تعیین می‌شود

## HMM (۱۵ حالت تاکتیک): π، A و B
- حالت‌ها = ۱۵ تاکتیک. شروع کار با **MLE نظارتی** از زنجیره‌های train (حالت‌ها در train معلوم‌اند) و سپس (در صورت وجود PyTorch) fine-tune با Adam روی NLL تکنیک بعدی.
- `A`: گذار بین تکنیک‌های متوالی زنجیره (برای forward filtering روی prefix).
- `A_next`: گذار «یک گام جلوتر» از تاکتیک آخرین ورودی به تاکتیک target، که نظارتی از نمونه‌های train برآورد می‌شود. دلیل: در این دیتاست target همیشه در تاکتیک **بعدی** است، ولی گذار زنجیره‌ای عمدتاً self-transition است.
- پیش‌بینی با **forward filtering** روی پنجره: `P(x_next | prefix) = Σ_s' (α_T A_next)(s') · B(s', x_next)`.
- **قید یکسان برای هر سه مدل:** تکنیکی که در پنجرهٔ ورودی آمده هرگز target نمی‌شود (assert گام ۱)؛ احتمال آن صفر و بقیه renormalize می‌شود (`MASK_SEEN`).
- `π_trunc`: توزیع آغازین جدا برای پنجره‌هایی که از وسط زنجیره بریده شده‌اند.

In [ ]:
# ============================== HMM (NumPy): پارامترها، MLE، forward، Viterbi ==============================
class HMMParams:
    def __init__(self, pi, pi_tr, A, B, A_next):
        self.pi, self.pi_tr, self.A, self.B, self.A_next = (np.asarray(x, np.float64) for x in (pi, pi_tr, A, B, A_next))
    def arrays(self):
        return dict(pi=self.pi, pi_tr=self.pi_tr, A=self.A, B=self.B, A_next=self.A_next)
    def n_params(self):
        return sum(a.size for a in self.arrays().values())
    def check(self, atol=1e-6):
        for k, a in self.arrays().items():
            assert np.isfinite(a).all() and (a >= 0).all(), f"HMM.{k} نامعتبر"
        assert abs(self.pi.sum() - 1) < atol and abs(self.pi_tr.sum() - 1) < atol
        assert np.allclose(self.A.sum(1), 1, atol=atol) and np.allclose(self.B.sum(1), 1, atol=atol) and np.allclose(self.A_next.sum(1), 1, atol=atol)
        return True

def hmm_mle(vocab, tr_idx):
    cA = np.full((NT, NT), CFG["A_SMOOTH"]); cB = np.full((NT, vocab.n), CFG["B_SMOOTH"])
    cpi = np.full(NT, CFG["PI_SMOOTH"]); cpt = np.full(NT, CFG["PI_SMOOTH"])
    for c in np.unique(CAMP_IDX[tr_idx]):                                   # فقط کمپین‌های train
        v = vocab.map[CHAIN[c]]; v = v[v >= 0]
        if len(v) == 0:
            continue
        s = vocab.tact[v]
        cpi[s[0]] += 1; np.add.at(cpt, s, 1.0); np.add.at(cB, (s, v), 1.0)
        if len(v) > 1:
            np.add.at(cA, (s[:-1], s[1:]), 1.0)
    cAn = np.full((NT, NT), CFG["A_SMOOTH"])                                # A_next: تاکتیک آخرین ورودی -> تاکتیک target
    np.add.at(cAn, (TACT_OF_GID[GX[tr_idx, GLEN[tr_idx] - 1]], GYT[tr_idx]), 1.0)
    return HMMParams(cpi / cpi.sum(), cpt / cpt.sum(), cA / cA.sum(1, keepdims=True), cB / cB.sum(1, keepdims=True),
                     cAn / cAn.sum(1, keepdims=True))

def apply_seen_np(p, tok, lens):
    """احتمال تکنیک‌های موجود در پنجرهٔ ورودی = 0 و renormalize (فقط از اطلاعات ورودی)."""
    if not CFG["MASK_SEEN"]:
        return p
    valid = np.arange(tok.shape[1])[None, :] < lens[:, None]
    m = np.zeros((len(tok), p.shape[1] + 2), bool)
    np.put_along_axis(m, np.where(valid, tok, 0), True, axis=1)
    q = np.where(m[:, 2:], 0.0, p); s = q.sum(1, keepdims=True)
    return np.where(s > 1e-300, q / np.maximum(s, 1e-300), p)

def hmm_predict(P, tok, lens, trunc, return_alpha=False):
    """forward filtering برداری روی batch؛ خروجی (N, n_classes)."""
    Bn, T = tok.shape; Bt = P.B.T
    alpha = np.zeros((Bn, NT))
    for t in range(T):
        valid = t < lens
        if not valid.any():
            break
        v = tok[:, t] - 2
        lik = np.where((v >= 0)[:, None], Bt[np.maximum(v, 0)], 1.0)          # UNK/PAD = بدون شواهد
        pred = np.where(trunc[:, None], P.pi_tr[None], P.pi[None]) if t == 0 else alpha @ P.A
        new = pred * lik; new /= new.sum(1, keepdims=True)
        alpha = np.where(valid[:, None], new, alpha)
    p = (alpha @ P.A_next) @ P.B
    p /= p.sum(1, keepdims=True)
    p = apply_seen_np(p, tok, lens)
    return (p, alpha) if return_alpha else p

def viterbi_np(P, v):
    """Viterbi گذشته‌نگر (diagnostic) روی یک زنجیرهٔ کامل؛ پیش‌بینی آینده نیست."""
    T = len(v); logA = np.log(P.A); logB = np.log(P.B[:, v])
    delta = np.log(P.pi) + logB[:, 0]; back = np.zeros((T, NT), np.int64)
    for t in range(1, T):
        sc = delta[:, None] + logA; back[t] = sc.argmax(0); delta = sc.max(0) + logB[:, t]
    path = np.zeros(T, np.int64); path[-1] = delta.argmax()
    for t in range(T - 1, 0, -1):
        path[t - 1] = back[t, path[t]]
    return path

# تست سریع
_p = hmm_mle(_v, FOLDS[0][0]); _p.check()
_pp = hmm_predict(_p, _tok, _ln, _tr)
assert _pp.shape == (len(_tok), _v.n) and np.allclose(_pp.sum(1), 1)
print("HMM-MLE ok | params:", _p.n_params())

## VOHMM — جزء VOM (Variable-Order Markov)
- contextها از order ۱ تا `WINDOW`، **longest-seen context + back-off** با *interpolated absolute discounting* و smoothing (`D` و `eps`).
- ترکیب: `P_VOHMM = α·P_VOM + (1−α)·P_HMM`.

In [ ]:
# ============================== VOM ==============================
class VOM:
    def __init__(self, n_cls, max_order, eps=1e-3):
        self.C, self.K, self.eps = n_cls, max_order, eps
    def fit(self, tok, lens, y):
        self.uni = np.bincount(y, minlength=self.C).astype(np.float64)
        raw = [dict() for _ in range(self.K + 1)]
        for i in range(len(y)):
            li = int(lens[i]); seq = tok[i, :li].tolist(); yi = int(y[i])
            for k in range(1, min(self.K, li) + 1):
                d = raw[k].setdefault(tuple(seq[li - k:]), {})
                d[yi] = d.get(yi, 0) + 1
        self.tab = [None] + [
            {ctx: (np.fromiter(d.keys(), np.int64, len(d)), np.fromiter(d.values(), np.float64, len(d))) for ctx, d in raw[k].items()}
            for k in range(1, self.K + 1)]
        self.tab = [None] + [{c: (ys, ns, ns.sum()) for c, (ys, ns) in t.items()} for t in self.tab[1:]]
        return self
    def predict(self, tok, lens, D=0.8):
        N = len(lens); P = np.empty((N, self.C))
        p0 = (self.uni + self.eps) / (self.uni.sum() + self.eps * self.C)
        self.last_order = np.zeros(N, np.int32)
        for i in range(N):
            li = int(lens[i]); seq = tok[i, :li].tolist(); p = p0
            for k in range(1, min(self.K, li) + 1):
                e = self.tab[k].get(tuple(seq[li - k:]))
                if e is None:
                    break                                    # context بلندتر دیده نشده؛ back-off به order قبلی
                ys, ns, tot = e
                q = p * (D * len(ys) / tot); q[ys] += np.maximum(ns - D, 0.0) / tot
                p = q; self.last_order[i] = k
            P[i] = p
        return apply_seen_np(P, tok, lens)
    def fingerprint(self):
        h = hashlib.sha256(self.uni.tobytes())
        for k in range(1, self.K + 1):
            t = self.tab[k]
            h.update(np.array([len(t), sum(e[2] for e in t.values()), sum(float((e[0] * e[1]).sum()) for e in t.values())]).tobytes())
        return h.hexdigest()

_tr_idx, _te_idx = FOLDS[0]
_tt, _tl, _ = _v.encode(_tr_idx); _ty = _v.target(_tr_idx)
_vom = VOM(_v.n, L, CFG["VOM_EPS"]).fit(_tt, _tl, _ty)
_pv = _vom.predict(_tok[:50], _ln[:50], 0.8)
assert np.allclose(_pv.sum(1), 1) and (_pv >= 0).all()
print("VOM ok | contexts per order:", [len(t) for t in _vom.tab[1:6]], "... | mean longest-seen order:", _vom.last_order.mean().round(2))

## متریک‌ها
Top-1/3/5، MRR، Target Coverage، متریک‌های closed-vocabulary (فقط targetهای موجود در vocab آموزش)، Group-macro MRR، NLL و ROC-AUC چندکلاسه (OVR macro).
- در **open-vocabulary** targetِ خارج از vocab شکست حساب می‌شود (MRR=0).
- رتبه با رویکرد محافظه‌کارانه محاسبه می‌شود (tieها به ضرر مدل).

In [ ]:
# ============================== متریک‌ها ==============================
def rank_of_target(P, y):
    pt = P[np.arange(len(y)), y][:, None]
    return (P >= pt).sum(1).astype(np.int32)                 # = 1 + #بزرگتر + #مساوی‌های دیگر (pessimistic)

def auc_ovr_macro(P, y):
    """ROC-AUC چندکلاسهٔ OVR macro با آمارهٔ Mann-Whitney (برداری)؛ فقط کلاس‌هایی که هم مثبت و هم منفی دارند."""
    N, C = P.shape
    pos = np.bincount(y, minlength=C); neg = N - pos
    ok = (pos > 0) & (neg > 0)
    if not ok.any():
        return np.nan, np.array([]), np.array([], int)
    R = rankdata(P[:, ok], axis=0)
    Y = np.zeros((N, C), bool); Y[np.arange(N), y] = True
    n1, n0 = pos[ok].astype(float), neg[ok].astype(float)
    auc = ((R * Y[:, ok]).sum(0) - n1 * (n1 + 1) / 2) / (n1 * n0)
    return float(auc.mean()), auc, np.flatnonzero(ok)

def compute_metrics(P, y, grp, full=True):
    n = len(y); cov = y >= 0
    m = {"n": n, "n_closed": int(cov.sum()), "coverage": float(cov.mean())}
    rank = np.zeros(n, np.int32); ptgt = np.full(n, np.nan)
    if cov.any():
        yc, Pc = y[cov], P[cov]
        rank[cov] = rank_of_target(Pc, yc); ptgt[cov] = Pc[np.arange(len(yc)), yc]
    rr = np.where(cov, 1.0 / np.maximum(rank, 1), 0.0)
    for k in (1, 3, 5):
        hit = cov & (rank <= k)
        m[f"top{k}_open"] = float(hit.mean()); m[f"top{k}_closed"] = float(hit[cov].mean()) if cov.any() else np.nan
    m["mrr_open"] = float(rr.mean()); m["mrr_closed"] = float(rr[cov].mean()) if cov.any() else np.nan
    nll = -np.log(np.clip(ptgt[cov], 1e-12, None)).mean() if cov.any() else np.nan
    m["nll_closed"] = float(nll); m["ppl_closed"] = float(np.exp(nll))
    dfg = pd.DataFrame({"g": grp, "rr": rr, "cov": cov})
    m["gmrr_open"] = float(dfg.groupby("g")["rr"].mean().mean())
    m["gmrr_closed"] = float(dfg[dfg["cov"]].groupby("g")["rr"].mean().mean()) if cov.any() else np.nan
    if full:
        a, per_cls, cls = auc_ovr_macro(P[cov], y[cov]) if cov.any() else (np.nan, None, None)
        m["auc_macro"] = a
        return m, rank, rr, ptgt, (per_cls, cls)
    return m, rank, rr, ptgt, None

def group_table(grp, rank, rr):
    d = pd.DataFrame({"g": grp, "rr": rr, "t1": (rank > 0) & (rank <= 1), "t3": (rank > 0) & (rank <= 3), "t5": (rank > 0) & (rank <= 5)})
    return d.groupby("g").agg(n=("rr", "size"), sum_rr=("rr", "sum"), mrr=("rr", "mean"),
                              top1=("t1", "mean"), top3=("t3", "mean"), top5=("t5", "mean"))

# تست: مدل کامل = 1 ، یکنواخت ≈ تصادفی
_C = 20; _yy = np.random.default_rng(0).integers(0, _C, 400); _g = np.arange(400) % 7
_P1 = np.full((400, _C), 1e-3); _P1[np.arange(400), _yy] = 1.0
_m1 = compute_metrics(_P1, _yy, _g)[0]; assert _m1["top1_open"] == 1 and _m1["mrr_open"] == 1 and _m1["auc_macro"] > 0.999
_m0 = compute_metrics(np.random.default_rng(1).random((400, _C)), _yy, _g)[0]; assert 0.4 < _m0["auc_macro"] < 0.6
print("metrics ok | random AUC=%.3f" % _m0["auc_macro"])

## SHHMM (PyTorch + CUDA)
**معماری:** embedding تکنیک + embedding تاکتیک ← GRU (علّی، ۲ لایه) ← fusion گیت‌دار با posterior حالت‌های تاکتیک HMM ← سر اصلی (تکنیک بعدی) + سر کمکی (تاکتیک بعدی).

**تابع هزینه:** `CE وزن‌دار (class-balanced) + λ_aux·CE(تاکتیک بعدی) + λ_hmm·NLL(HMM) + λ_state·CE(حالت تاکتیک)`.

**آموزش روی دیدهای علّی augment‌شده:** به‌هم‌ریختن ترتیب درون هر تاکتیک (ترتیب درون تاکتیک در دیتاست فقط الفبایی است)، بریدن تاریخچهٔ قدیمی، و حذف تصادفی توکن‌ها (→ UNK). همه روی GPU و برداری؛ فقط روی train و فقط گذشته (علّی).

**کارایی:** دادهٔ کامل روی GPU، بدون DataLoader، bf16 autocast برای GRU/Linear (HMM همیشه fp32)، TF32، `cudnn.benchmark`، فقط یک sync در هر epoch.

In [ ]:
# ============================== مدل‌های PyTorch ==============================
if HAS_TORCH:
    class TorchHMM(nn.Module):
        def __init__(self, p):
            super().__init__()
            lg = lambda a: nn.Parameter(torch.tensor(np.log(np.maximum(a, 1e-12)), dtype=torch.float32))
            self.pi_l, self.pit_l, self.A_l, self.B_l, self.An_l = lg(p.pi), lg(p.pi_tr), lg(p.A), lg(p.B), lg(p.A_next)
        def forward(self, tok, lens, trunc, return_steps=False):
            with torch.autocast(device_type=tok.device.type, enabled=False):      # HMM همیشه fp32
                Bsz, T = tok.shape
                A = F.softmax(self.A_l, 1); Bm = F.softmax(self.B_l, 1); An = F.softmax(self.An_l, 1)
                pi = F.softmax(self.pi_l, 0); pit = F.softmax(self.pit_l, 0)
                v = tok - 2
                lik = Bm.t().contiguous()[v.clamp_min(0)]                          # (B,T,S) یک gather برای همهٔ گام‌ها
                lik = torch.where((v >= 0).unsqueeze(-1), lik, torch.ones_like(lik))
                pos_mask = torch.arange(T, device=tok.device)[None, :] < lens[:, None]
                init = torch.where(trunc.unsqueeze(-1), pit.unsqueeze(0), pi.unsqueeze(0))
                alpha, steps = init, []
                for t in range(T):
                    pred = init if t == 0 else alpha @ A
                    new = pred * lik[:, t]
                    new = new / new.sum(-1, keepdim=True).clamp_min(1e-30)
                    alpha = torch.where(pos_mask[:, t:t + 1], new, alpha)
                    if return_steps:
                        steps.append(alpha)
                p_next = (alpha @ An) @ Bm
                return alpha, p_next, (torch.stack(steps, 1) if return_steps else None)
        def export(self):
            with torch.no_grad():
                sm = lambda x, d: F.softmax(x, d).double().cpu().numpy()
                return HMMParams(sm(self.pi_l, 0), sm(self.pit_l, 0), sm(self.A_l, 1), sm(self.B_l, 1), sm(self.An_l, 1))

    class SHHMM(nn.Module):
        def __init__(self, n_cls, tact_tok, hmm_params, d=128, h=256, layers=2, drop=0.2):
            super().__init__()
            self.hmm = TorchHMM(hmm_params)
            self.emb = nn.Embedding(n_cls + 2, d, padding_idx=0)
            self.temb = nn.Embedding(NT + 1, d)
            self.register_buffer("tact_tok", torch.as_tensor(tact_tok, dtype=torch.long))
            self.in_drop = nn.Dropout(0.1)
            self.gru = nn.GRU(d, h, num_layers=layers, batch_first=True, dropout=drop if layers > 1 else 0.0)
            self.ln = nn.LayerNorm(h)
            self.post = nn.Linear(NT, h)
            self.gate = nn.Linear(2 * h, h)
            self.drop = nn.Dropout(0.3)
            self.out = nn.Linear(h, n_cls)
            self.aux = nn.Linear(h, NT)
        def forward(self, tok, lens, trunc, return_steps=False):
            alpha, p_hmm, steps = self.hmm(tok, lens, trunc, return_steps)
            x = self.in_drop(self.emb(tok) + self.temb(self.tact_tok[tok]))
            out, _ = self.gru(x)                                                  # راست‌پد: خروجی گام len-1 دقیق است
            hl = out[torch.arange(tok.size(0), device=tok.device), lens - 1]
            u = self.ln(hl); z = self.post(alpha)
            g = torch.sigmoid(self.gate(torch.cat([u, z.to(u.dtype)], -1)))
            f = self.drop(g * u + (1 - g) * z.to(u.dtype))
            return self.out(f), self.aux(f), p_hmm, steps

    def augment_views(tok, lens, trunc, tact_tok, p_view, p_shuffle, p_trunc, p_drop):
        """دیدهای علّی augment‌شده (برداری روی GPU). target تغییر نمی‌کند و فقط گذشته دستکاری می‌شود."""
        B, T = tok.shape; dev = tok.device
        pos = torch.arange(T, device=dev)[None, :]
        valid = pos < lens[:, None]
        tk = tact_tok[tok].float()
        sel = torch.rand(B, 1, device=dev) < p_shuffle
        noise = torch.where(sel, torch.rand(B, T, device=dev), pos.float().expand(B, T) / T)
        key = torch.where(valid, tk + 0.99 * noise, 1e4 + pos.float().expand(B, T))
        tok2 = tok.gather(1, key.argsort(1))                                      # شافل درون تاکتیک
        do_tr = torch.rand(B, device=dev) < p_trunc
        m = torch.minimum((torch.rand(B, device=dev) * lens.float()).floor().long() + 1, lens)
        new_len = torch.where(do_tr, m, lens); start = lens - new_len
        tok3 = tok2.gather(1, (pos + start[:, None]).clamp_max(T - 1))
        valid3 = pos < new_len[:, None]
        tok3 = torch.where(valid3, tok3, torch.zeros_like(tok3))
        drop = (torch.rand(B, T, device=dev) < p_drop) & valid3 & (pos != (new_len - 1)[:, None])
        tok3 = torch.where(drop, torch.ones_like(tok3), tok3)                    # UNK
        use = torch.rand(B, device=dev) < p_view
        return (torch.where(use[:, None], tok3, tok), torch.where(use, new_len, lens),
                torch.where(use, trunc | (start > 0), trunc))

    def seen_mask_t(tok, lens, n_cls):
        valid = torch.arange(tok.size(1), device=tok.device)[None, :] < lens[:, None]
        m = torch.zeros(tok.size(0), n_cls + 2, device=tok.device)
        m.scatter_(1, torch.where(valid, tok, torch.zeros_like(tok)), 1.0)
        return m[:, 2:] > 0

    def mask_probs_t(p, seen):
        q = p.masked_fill(seen, 0.0)
        return q / q.sum(1, keepdim=True).clamp_min(1e-12)

    def to_dev_tensors(tok, lens, trunc, y=None, yt=None):
        T = dict(tok=torch.as_tensor(tok, device=DEVICE), lens=torch.as_tensor(lens, device=DEVICE),
                 trunc=torch.as_tensor(trunc, device=DEVICE))
        if y is not None:
            T["y"] = torch.as_tensor(y, device=DEVICE); T["yt"] = torch.as_tensor(yt, device=DEVICE)
        return T

    def state_loss(steps, tok, lens, tact_tok):
        T = tok.size(1)
        tk = tact_tok[tok]
        valid = (tk < NT) & (torch.arange(T, device=tok.device)[None, :] < lens[:, None])
        lp = steps.clamp_min(1e-12).log().gather(2, tk.clamp_max(NT - 1).unsqueeze(-1)).squeeze(-1)
        return -(lp * valid).sum() / valid.sum().clamp_min(1)

    def run_training(model, kind, T, vocab, lr, epochs, bs, wd, seed):
        dev = DEVICE; n = T["tok"].size(0); spe = math.ceil(n / bs)
        total = max(1, epochs * spe); warm = max(1, int(0.05 * total))
        cw = torch.as_tensor(class_weight_vector(vocab.counts, CFG["IMBALANCE"], CFG["EFF_BETA"]), dtype=torch.float32, device=dev)
        lpr = torch.as_tensor(log_prior(vocab.counts), dtype=torch.float32, device=dev)
        tact_tok = torch.as_tensor(vocab.tact_tok, device=dev)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        sched = torch.optim.lr_scheduler.LambdaLR(
            opt, lambda s: min(1.0, (s + 1) / warm) * (0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total)))))
        gen = torch.Generator(device=dev); gen.manual_seed(seed)
        aug = CFG["AUG"]; ls = CFG["LABEL_SMOOTH"]; hist = []
        for ep in range(epochs):
            model.train(); perm = torch.randperm(n, device=dev, generator=gen)
            acc = torch.zeros(3, device=dev)
            for i in range(0, n, bs):
                idx = perm[i:i + bs]
                tok, ln, tr, y, yt = T["tok"][idx], T["lens"][idx], T["trunc"][idx], T["y"][idx], T["yt"][idx]
                if kind == "shhmm":
                    tok, ln, tr = augment_views(tok, ln, tr, tact_tok, **aug)
                seen = seen_mask_t(tok, ln, vocab.n) if CFG["MASK_SEEN"] else None
                with torch.autocast(device_type=dev.type, dtype=torch.bfloat16, enabled=USE_AMP):
                    if kind == "hmm":
                        _, p, steps = model(tok, ln, tr, True)
                        if seen is not None:
                            p = mask_probs_t(p, seen)
                        lp = p.clamp_min(1e-12).log()
                        loss_extra = CFG["LAM_STATE"] * state_loss(steps, tok, ln, tact_tok)
                        sm_term = 0.0
                    else:
                        logits, aux, p_h, steps = model(tok, ln, tr, True)
                        logits = logits.float()
                        if seen is not None:
                            logits = logits.masked_fill(seen, -1e4); p_h = mask_probs_t(p_h, seen)
                        lp = F.log_softmax(logits, -1)
                        sm_term = -lp.mean(-1)
                        nll_h = -p_h.clamp_min(1e-12).log().gather(1, y[:, None]).squeeze(1).mean()
                        loss_extra = (CFG["LAM_AUX"] * F.cross_entropy(aux.float(), yt)
                                      + CFG["LAM_HMM"] * nll_h + CFG["LAM_STATE"] * state_loss(steps, tok, ln, tact_tok))
                    plain = -lp.gather(1, y[:, None]).squeeze(1)
                    if CFG["IMBALANCE"] == "logit_adjust":
                        plain_train = -F.log_softmax(lp + CFG["LOGIT_TAU"] * lpr, -1).gather(1, y[:, None]).squeeze(1)
                    else:
                        plain_train = plain
                    ce = (1 - ls) * plain_train + ls * sm_term if kind == "shhmm" else plain_train
                    loss = (cw[y] * ce).mean() + loss_extra
                opt.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step(); sched.step()
                acc[0] += loss.detach() * len(idx); acc[1] += plain.detach().sum(); acc[2] += len(idx)
            a = acc.tolist()                                                        # تنها sync در هر epoch
            hist.append(dict(epoch=ep + 1, train_loss=a[0] / a[2], train_nll=a[1] / a[2], lr=sched.get_last_lr()[0]))
        model.eval()
        assert all(torch.isfinite(p).all() for p in model.parameters()), "NaN/Inf در پارامترها"
        return hist

    @torch.inference_mode()
    def shhmm_predict_torch(model, tok, lens, trunc, bs=2048):
        model.eval(); outs = []
        for i in range(0, len(tok), bs):
            t_ = to_dev_tensors(tok[i:i + bs], lens[i:i + bs], trunc[i:i + bs])
            with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16, enabled=USE_AMP):
                logits = model(t_["tok"], t_["lens"], t_["trunc"])[0].float()
            if CFG["MASK_SEEN"]:
                logits = logits.masked_fill(seen_mask_t(t_["tok"], t_["lens"], logits.size(1)), -1e4)
            outs.append(F.softmax(logits, -1))
        P = torch.cat(outs).double().cpu().numpy()
        return P / P.sum(1, keepdims=True)
    print("PyTorch models defined (device=%s)" % DEVICE)
else:
    print("PyTorch غیرفعال: SHHMM تعریف نشد.")

In [ ]:
# ============================== fit / predict برای هر سه مدل ==============================
HMM_LR_GRID = CFG["HMM_LRS"] if HAS_TORCH else [None]
def _lr_key(lr): return NA if lr is None else float(lr)

def fit_hmm(vocab, tr_idx, lr, seed, mle=None):
    p0 = mle if mle is not None else hmm_mle(vocab, tr_idx)
    if not HAS_TORCH or lr is None:
        return dict(params=p0, hist=[], lr=None)
    seed_all(seed)
    tok, ln, tr = vocab.encode(tr_idx)
    T = to_dev_tensors(tok, ln, tr, vocab.target(tr_idx), GYT[tr_idx])
    m = TorchHMM(p0).to(DEVICE)
    hist = run_training(m, "hmm", T, vocab, lr, CFG["HMM_EPOCHS"], CFG["HMM_BS"], 0.0, seed)
    return dict(params=m.export(), hist=hist, lr=lr)

def fit_shhmm(vocab, tr_idx, lr, seed, mle=None):
    p0 = mle if mle is not None else hmm_mle(vocab, tr_idx)
    seed_all(seed)
    tok, ln, tr = vocab.encode(tr_idx)
    T = to_dev_tensors(tok, ln, tr, vocab.target(tr_idx), GYT[tr_idx])
    m = SHHMM(vocab.n, vocab.tact_tok, p0, CFG["SH_EMB"], CFG["SH_HID"], CFG["SH_LAYERS"], CFG["SH_DROP"]).to(DEVICE)
    hist = run_training(m, "shhmm", T, vocab, lr, CFG["SH_EPOCHS"], CFG["SH_BS"], CFG["SH_WD"], seed)
    return dict(model=m, hist=hist, lr=lr)

def blend(Pa, Pb, alpha):
    return alpha * Pa + (1.0 - alpha) * Pb

In [ ]:
# ============================== تست‌های یکپارچگی و نشت (قبل از اجرای CV) ==============================
# 1) پیش‌بینی HMM جمع احتمال = 1 و تکنیک‌های UNK بی‌اثرند
_ptest = hmm_predict(_p, _tok, _ln, _tr)
_tok_unk = _tok.copy(); _tok_unk[:, 0] = 1                                     # UNK در گام اول
assert np.allclose(hmm_predict(_p, _tok_unk, _ln, _tr).sum(1), 1)

# 2) علّیت: تغییر توکن‌های بعد از طول واقعی (پد) خروجی را عوض نمی‌کند
_tokc = _tok.copy(); _mask = np.arange(_tok.shape[1])[None, :] >= _ln[:, None]
_tokc[_mask] = np.random.default_rng(0).integers(2, _v.n + 2, _mask.sum())
assert np.allclose(hmm_predict(_p, _tokc, _ln, _tr), _ptest)
_vp1 = _vom.predict(_tok[:100], _ln[:100]); _vp2 = _vom.predict(_tokc[:100], _ln[:100])
assert np.allclose(_vp1, _vp2)

# 2b) قید no-repeat: احتمال تکنیک‌های داخل ورودی صفر است و جمع احتمال ۱ می‌ماند
_seen_ok = np.zeros(len(_tok), bool)
for _i in range(len(_tok)):
    _s = _tok[_i, :_ln[_i]] - 2; _s = _s[_s >= 0]
    _seen_ok[_i] = (_ptest[_i, _s] == 0).all() if CFG["MASK_SEEN"] else True
assert _seen_ok.all() and np.allclose(_ptest.sum(1), 1)

# 3) VOM نشتی ندارد: آموزش روی train فقط؛ نمونهٔ test در جدول‌ها اضافه نشده است
_n_ctx_before = sum(len(t) for t in _vom.tab[1:])
_vom_leaky = VOM(_v.n, L, CFG["VOM_EPS"]).fit(*_v.encode(np.concatenate([_tr_idx, _te_idx]))[:2], _v.target(np.concatenate([_tr_idx, _te_idx])).clip(0))
assert sum(len(t) for t in _vom_leaky.tab[1:]) > _n_ctx_before, "تست حساسیت: ورود test باید جدول را تغییر دهد"

# 4) تکراری‌های (input,target) مشترک بین train و test (فقط گزارش؛ از گروه‌های مختلف می‌آیند)
_k = lambda idx: {(tuple(SAMPLES[i]["input"]), SAMPLES[i]["target"]) for i in idx}
_dup = len(_k(_tr_idx) & _k(_te_idx)) / max(1, len(_k(_te_idx)))
print(f"fold0: (input,target) مشترک train/test = {_dup:.2%} (از گروه‌های متفاوت؛ نشت گروهی نیست)")

# 5) تست‌های PyTorch (اگر موجود باشد)
if HAS_TORCH:
    _th = TorchHMM(_p).to(DEVICE).eval()
    _t_ = to_dev_tensors(_tok[:64], _ln[:64], _tr[:64])
    with torch.no_grad():
        _, _pt, _st = _th(_t_["tok"], _t_["lens"], _t_["trunc"], True)
    _ms = CFG["MASK_SEEN"]; CFG["MASK_SEEN"] = False
    _ref = hmm_predict(_p, _tok[:64], _ln[:64], _tr[:64]); CFG["MASK_SEEN"] = _ms
    assert np.abs(_pt.double().cpu().numpy() - _ref).max() < 1e-4, "HMM torch != numpy"
    assert np.abs(_th.export().A - _p.A).max() < 1e-5 and np.abs(_th.export().A_next - _p.A_next).max() < 1e-5
    _sm = seen_mask_t(_t_["tok"], _t_["lens"], _v.n); assert (mask_probs_t(_pt, _sm)[_sm] == 0).all()
    _sh = SHHMM(_v.n, _v.tact_tok, _p).to(DEVICE).eval()
    with torch.no_grad():
        _o1 = _sh(_t_["tok"], _t_["lens"], _t_["trunc"])[0]
        _tk2 = _t_["tok"].clone(); _m2 = torch.arange(L, device=DEVICE)[None, :] >= _t_["lens"][:, None]
        _tk2[_m2] = torch.randint(2, _v.n + 2, (int(_m2.sum()),), device=DEVICE)
        _o2 = _sh(_tk2, _t_["lens"], _t_["trunc"])[0]
    assert torch.allclose(_o1, _o2, atol=1e-4), "SHHMM علّی نیست"
    _sh.train(); _lg, _ax, _ph, _stp = _sh(_t_["tok"], _t_["lens"], _t_["trunc"], True)
    (_lg.logsumexp(-1).mean() + _ax.logsumexp(-1).mean() + _ph.log().mean() + _stp.log().mean()).backward()
    assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in _sh.parameters()), "گرادیان نامعتبر"
    _tt_ = torch.as_tensor(_v.tact_tok, device=DEVICE)
    _a_tok, _a_len, _a_tr = augment_views(_t_["tok"], _t_["lens"], _t_["trunc"], _tt_, 1.0, 1.0, 0.5, 0.1)
    assert (_a_len >= 1).all() and (_a_len <= L).all() and (_a_tok[torch.arange(L, device=DEVICE)[None, :] >= _a_len[:, None]] == 0).all()
    print("PyTorch self-tests ok: HMM==numpy, SHHMM causal, gradients finite, augmentation shapes")
print("✓ تست‌های یکپارچگی و نشت گذشت")

## اجرای ۱۰-fold GroupKFold روی Development
برای هر fold: آموزش HMM (به‌ازای هر LR)، VOM، و SHHMM (به‌ازای هر LR) فقط روی train، سپس ارزیابی همهٔ ترکیب‌های `α` و `D` روی fold کنار گذاشته‌شده. شرایط آزمون برای هر سه مدل یکسان است (همان نمونه‌ها، همان vocabulary، همان متریک‌ها).

In [ ]:
# ============================== 10-fold GroupKFold ==============================
MODELS = ["HMM", "SHHMM", "VOHMM"] if HAS_TORCH else ["HMM", "VOHMM"]
CV_ROWS, OOF = [], {}
TF_OOF = np.zeros(N_S)                                    # فراوانی train برای target هر نمونه (در fold خودش)
KEYC = ["model", "lr_h", "lr_s", "D", "alpha"]

def _record(key, fold, te, m, rank, rr, ptgt):
    CV_ROWS.append(dict(zip(KEYC, key), fold=fold, **m))
    o = OOF.setdefault(key, dict(rank=np.zeros(N_S, np.int32), rr=np.zeros(N_S), ptgt=np.full(N_S, np.nan)))
    o["rank"][te], o["rr"][te], o["ptgt"][te] = rank, rr, ptgt

def eval_all_configs(P_hmm, P_vom, P_neu, y, grp, fold, te):
    def go(key, P):
        m, rank, rr, pt, _ = compute_metrics(P, y, grp, full=True)
        _record(key, fold, te, m, rank, rr, pt)
    for lr, P in P_hmm.items():
        go(("HMM", _lr_key(lr), NA, NA, NA), P)
    for D, Pv in P_vom.items():
        for a in CFG["ALPHAS"]:
            if a == 1.0:
                go(("VOHMM", NA, NA, D, 1.0), Pv)
            else:
                for lr, Ph in P_hmm.items():
                    go(("VOHMM", _lr_key(lr), NA, D, a), blend(Pv, Ph, a))
    for ls, Pn in P_neu.items():
        for a in CFG["ALPHAS"]:
            if a == 1.0:
                go(("SHHMM", NA, ls, NA, 1.0), Pn)
            else:
                for lr, Ph in P_hmm.items():
                    go(("SHHMM", _lr_key(lr), ls, NA, a), blend(Pn, Ph, a))

t_cv = time.time()
for f, (tr, te) in enumerate(FOLDS):
    t0 = time.time(); seed = CFG["SEED"] + f
    vocab = Vocab(tr)
    tok_tr, ln_tr, _ = vocab.encode(tr); y_tr = vocab.target(tr)
    tok_te, ln_te, trc_te = vocab.encode(te); y_te = vocab.target(te)
    TF_OOF[te] = vocab.counts[np.maximum(y_te, 0)] * (y_te >= 0)
    mle = hmm_mle(vocab, tr)
    P_hmm = {lr: hmm_predict(fit_hmm(vocab, tr, lr, seed, mle)["params"], tok_te, ln_te, trc_te) for lr in HMM_LR_GRID}
    vom = VOM(vocab.n, L, CFG["VOM_EPS"]).fit(tok_tr, ln_tr, y_tr)
    P_vom = {D: vom.predict(tok_te, ln_te, D) for D in CFG["VOM_D"]}
    P_neu = {}
    if HAS_TORCH:
        for lr in CFG["SH_LRS"]:
            fit = fit_shhmm(vocab, tr, lr, seed, mle)
            P_neu[lr] = shhmm_predict_torch(fit["model"], tok_te, ln_te, trc_te)
            del fit
    eval_all_configs(P_hmm, P_vom, P_neu, y_te, GROUP_IDX[te], f, te)
    print(f"fold {f+1:2d}/{len(FOLDS)}  train={len(tr)} test={len(te)} vocab={vocab.n} cover={np.mean(y_te>=0):.3f}  [{time.time()-t0:.1f}s]")
print(f"CV total: {time.time()-t_cv:.1f}s | configs evaluated: {len(OOF)}")

In [ ]:
# ============================== انتخاب LR / α / D بر اساس میانگین CV ==============================
cv_df = pd.DataFrame(CV_ROWS)
MET_COLS = [c for c in cv_df.columns if c not in KEYC + ["fold", "n", "n_closed"]]
CV_AGG = cv_df.groupby(KEYC)[MET_COLS].mean().reset_index()
SEL = CFG["SELECT_METRIC"]

BEST, BEST_KEY, SEL_SCORE = {}, {}, {}
for m in MODELS:
    sub = CV_AGG[CV_AGG["model"] == m].sort_values(SEL, ascending=False, kind="stable").iloc[0]
    BEST[m] = dict(lr_h=float(sub.lr_h), lr_s=float(sub.lr_s), D=float(sub.D), alpha=float(sub.alpha))
    SEL_SCORE[m] = float(sub[SEL])
    BEST_KEY[m] = tuple([m] + [float(sub[c]) for c in KEYC[1:]])
BEST_MODEL = max(SEL_SCORE, key=SEL_SCORE.get)

print("بهترین پیکربندی هر مدل (بر اساس میانگین %s در CV):" % SEL)
for m in MODELS:
    print(f"  {m:7s}", {k: (None if v == NA else v) for k, v in BEST[m].items()}, "->", round(SEL_SCORE[m], 4))
print("بهترین مدل کلی:", BEST_MODEL)

# جدول LR هر مدل (بهترین α/D به‌ازای هر LR)
rows = []
for m, col in (("HMM", "lr_h"), ("VOHMM", "lr_h"), ("SHHMM", "lr_s")):
    if m in MODELS:
        t = CV_AGG[(CV_AGG.model == m) & (CV_AGG[col] != NA)].groupby(col)[SEL].max()
        rows += [dict(model=m, lr=k, best_cv_score=v) for k, v in t.items()]
LR_TABLE = pd.DataFrame(rows); print(LR_TABLE.to_string(index=False)) if len(LR_TABLE) else print("(در حالت MLE، LR وجود ندارد)")

CV_AGG.to_csv(OUT2 / "cv_all_configs_mean.csv", index=False)
cv_df.to_csv(OUT2 / "cv_all_configs_per_fold.csv", index=False)
def cfg_mask(df, key):
    mk = df["model"] == key[0]
    for c, v in zip(KEYC[1:], key[1:]):
        mk &= np.isclose(df[c].astype(float), v)
    return mk.values
CV_SUMMARY = pd.DataFrame([dict(model=m, **{c: cv_df.loc[cfg_mask(cv_df, BEST_KEY[m]), c].agg(["mean", "std"]).round(4).tolist()
                                            for c in ["top1_open", "top3_open", "top5_open", "mrr_open", "gmrr_open", "auc_macro", "nll_closed"]})
                           for m in MODELS]).set_index("model")
print("\nCV (mean, std) برای بهترین پیکربندی هر مدل:"); print(CV_SUMMARY.to_string())

## آموزش نهایی روی کل Development و ارزیابی روی FINAL test
FINAL test **فقط یک بار** و فقط بعد از قفل شدن همهٔ انتخاب‌ها استفاده می‌شود. مدل‌ها با بهترین LR/α/D از CV روی کل Development بازآموزی می‌شوند.

In [ ]:
# ============================== آموزش نهایی + ارزیابی FINAL ==============================
assert "FINAL_DONE" not in globals(), "FINAL test فقط یک بار باید ارزیابی شود"
seed_f = CFG["SEED"] + 1000
VOC = Vocab(DEV_IDX)
tok_dev, ln_dev, _ = VOC.encode(DEV_IDX); y_dev = VOC.target(DEV_IDX)
tok_f, ln_f, trc_f = VOC.encode(FINAL_IDX); y_f = VOC.target(FINAL_IDX)
grp_f = GROUP_IDX[FINAL_IDX]; TF_FINAL = VOC.counts[np.maximum(y_f, 0)] * (y_f >= 0)
MLE_DEV = hmm_mle(VOC, DEV_IDX)

need_lr = {BEST["HMM"]["lr_h"]}
for m in ("VOHMM", "SHHMM"):
    if m in MODELS and BEST[m]["alpha"] < 1.0:
        need_lr.add(BEST[m]["lr_h"])
FIT_HMM = {lr: fit_hmm(VOC, DEV_IDX, None if lr == NA else lr, seed_f, MLE_DEV) for lr in need_lr}
for fit in FIT_HMM.values():
    fit["params"].check()
FIT_VOM = VOM(VOC.n, L, CFG["VOM_EPS"]).fit(tok_dev, ln_dev, y_dev)
FIT_SH = fit_shhmm(VOC, DEV_IDX, BEST["SHHMM"]["lr_s"], seed_f, MLE_DEV) if "SHHMM" in MODELS else None

def predict_model(name, tok, lens, trunc):
    b = BEST[name]
    if name == "HMM":
        return hmm_predict(FIT_HMM[b["lr_h"]]["params"], tok, lens, trunc)
    Pm = FIT_VOM.predict(tok, lens, b["D"]) if name == "VOHMM" else shhmm_predict_torch(FIT_SH["model"], tok, lens, trunc)
    if b["alpha"] >= 1.0:
        return Pm
    return blend(Pm, hmm_predict(FIT_HMM[b["lr_h"]]["params"], tok, lens, trunc), b["alpha"])

P_FINAL, FINAL_RES = {}, {}
for m in MODELS:
    P_FINAL[m] = predict_model(m, tok_f, ln_f, trc_f)
    met, rank, rr, pt, auc = compute_metrics(P_FINAL[m], y_f, grp_f, full=True)
    FINAL_RES[m] = dict(metrics=met, rank=rank, rr=rr, ptgt=pt, auc=auc)
FINAL_DONE = True

FINAL_TABLE = pd.DataFrame({m: FINAL_RES[m]["metrics"] for m in MODELS}).T[
    ["n", "n_closed", "coverage", "top1_open", "top3_open", "top5_open", "mrr_open", "top1_closed", "top3_closed", "top5_closed",
     "mrr_closed", "gmrr_open", "gmrr_closed", "nll_closed", "ppl_closed", "auc_macro"]]
FINAL_TABLE.to_csv(OUT2 / "final_test_metrics.csv")
print("FINAL test (%d نمونه، %d گروه):" % (len(FINAL_IDX), len(FINAL_GROUPS)))
print(FINAL_TABLE.drop(columns=["n", "n_closed"]).round(4).to_string())
for m in MODELS:
    assert np.allclose(P_FINAL[m].sum(1), 1, atol=1e-6) and np.isfinite(P_FINAL[m]).all()

## استنتاج آماری
- عملکرد در سطح گروه (Group-macro) و pooled.
- Bootstrap با ۱۰۰۰ تکرار روی **گروه‌ها** (واحد مستقل) برای بازهٔ اطمینان ۹۵٪.
- Paired sign-flip permutation با ۱۰۰۰ تکرار روی اختلاف MRR هر گروه، برای هر جفت مدل، با اصلاح Holm.
- یک بار روی FINAL test و یک بار روی پیش‌بینی‌های out-of-fold کل Development (گروه‌های بیشتر ⇒ توان آماری بالاتر).

In [ ]:
# ============================== Bootstrap و Sign-flip ==============================
def group_inference(rank_d, rr_d, grp, seed, B=1000, P=1000):
    ms = list(rank_d); tabs = {m: group_table(grp, rank_d[m], rr_d[m]) for m in ms}
    idx = tabs[ms[0]].index; G = len(idx)
    rng = np.random.default_rng(seed); bi = rng.integers(0, G, (B, G))
    ci_rows = []
    for m in ms:
        t = tabs[m]; n = t["n"].values.astype(float)
        stats_ = {"mrr_pooled": (t["sum_rr"].values, n), "top1": (t["top1"].values * n, n), "top3": (t["top3"].values * n, n),
                  "top5": (t["top5"].values * n, n)}
        for name, (s, nn_) in stats_.items():
            boot = s[bi].sum(1) / nn_[bi].sum(1)
            ci_rows.append(dict(model=m, metric=name, point=s.sum() / nn_.sum(), lo=np.percentile(boot, 2.5), hi=np.percentile(boot, 97.5)))
        boot = t["mrr"].values[bi].mean(1)
        ci_rows.append(dict(model=m, metric="mrr_group_macro", point=t["mrr"].mean(), lo=np.percentile(boot, 2.5), hi=np.percentile(boot, 97.5)))
    pairs = []
    for i in range(len(ms)):
        for j in range(i + 1, len(ms)):
            d = (tabs[ms[i]]["mrr"] - tabs[ms[j]]["mrr"]).values
            obs = d.mean(); signs = rng.choice([-1.0, 1.0], size=(P, G))
            p = (1 + (np.abs((signs * d).mean(1)) >= abs(obs) - 1e-15).sum()) / (P + 1)
            lo_hi = np.percentile(d[rng.integers(0, G, (B, G))].mean(1), [2.5, 97.5])
            pairs.append(dict(a=ms[i], b=ms[j], mean_diff_group_mrr=obs, ci_lo=lo_hi[0], ci_hi=lo_hi[1], p_perm=p))
    pr = pd.DataFrame(pairs).sort_values("p_perm").reset_index(drop=True)
    pr["p_holm"] = np.minimum(1.0, np.maximum.accumulate((len(pr) - np.arange(len(pr))) * pr["p_perm"].values))
    return pd.DataFrame(ci_rows), pr, tabs

CI_FINAL, PAIR_FINAL, GTAB_FINAL = group_inference(
    {m: FINAL_RES[m]["rank"] for m in MODELS}, {m: FINAL_RES[m]["rr"] for m in MODELS}, grp_f, CFG["SEED"], CFG["BOOT"], CFG["PERM"])
grp_dev = GROUP_IDX[DEV_IDX]
CI_OOF, PAIR_OOF, GTAB_OOF = group_inference(
    {m: OOF[BEST_KEY[m]]["rank"][DEV_IDX] for m in MODELS}, {m: OOF[BEST_KEY[m]]["rr"][DEV_IDX] for m in MODELS}, grp_dev,
    CFG["SEED"] + 1, CFG["BOOT"], CFG["PERM"])
for name, a, b in (("FINAL", CI_FINAL, PAIR_FINAL), ("OOF-Dev", CI_OOF, PAIR_OOF)):
    a.to_csv(OUT2 / f"stats_ci_{name}.csv", index=False); b.to_csv(OUT2 / f"stats_pairwise_{name}.csv", index=False)
    print(f"\n[{name}] Bootstrap 95% CI (group-level):"); print(a.round(4).to_string(index=False))
    print(f"[{name}] Paired sign-flip permutation ({CFG['PERM']}):"); print(b.round(4).to_string(index=False))

## تحلیل تکنیک‌های نادر، طول prefix، ماتریس درهم‌ریختگی و Viterbi

In [ ]:
# ============================== Rare / Prefix / Confusion / Viterbi ==============================
RARE_BINS = [(0, 0, "unseen"), (1, 4, "1-4"), (5, 19, "5-19"), (20, 59, "20-59"), (60, 10**9, ">=60")]
PREFIX_BINS = [(1, 3, "1-3"), (4, 7, "4-7"), (8, 15, "8-15"), (16, 31, "16-31"), (32, 63, "32-63"), (64, 10**9, ">=64")]
def bin_table(vals, bins, rank, rr):
    rows = []
    for lo, hi, name in bins:
        mk = (vals >= lo) & (vals <= hi); n = int(mk.sum())
        rows.append(dict(bin=name, n=n, top1=((rank > 0) & (rank <= 1))[mk].mean() if n else np.nan,
                         top5=((rank > 0) & (rank <= 5))[mk].mean() if n else np.nan, mrr=rr[mk].mean() if n else np.nan))
    return pd.DataFrame(rows)

RARE_OOF, PREFIX_OOF = {}, {}
for m in MODELS:
    o = OOF[BEST_KEY[m]]
    RARE_OOF[m] = bin_table(TF_OOF[DEV_IDX], RARE_BINS, o["rank"][DEV_IDX], o["rr"][DEV_IDX])
    PREFIX_OOF[m] = bin_table(ORIG_LEN[DEV_IDX], PREFIX_BINS, o["rank"][DEV_IDX], o["rr"][DEV_IDX])
    RARE_OOF[m].to_csv(OUT2 / f"rare_technique_{m}.csv", index=False); PREFIX_OOF[m].to_csv(OUT2 / f"prefix_length_{m}.csv", index=False)
print("Rare-technique (OOF, بهترین مدل):", BEST_MODEL); print(RARE_OOF[BEST_MODEL].round(3).to_string(index=False))
print("Prefix-length (OOF):", BEST_MODEL); print(PREFIX_OOF[BEST_MODEL].round(3).to_string(index=False))

# ---------- Confusion matrix مدل منتخب (FINAL، closed-vocab) ----------
cov_f = y_f >= 0
pred_f = P_FINAL[BEST_MODEL][cov_f].argmax(1); true_f = y_f[cov_f]
CM_RAW = np.zeros((VOC.n, VOC.n), np.int32); np.add.at(CM_RAW, (true_f, pred_f), 1)
pd.DataFrame(CM_RAW, index=VOC.names, columns=VOC.names).to_csv(OUT2 / "confusion_matrix_raw.csv")
top_cls = np.argsort(-np.bincount(true_f, minlength=VOC.n))[:CFG["CM_TOP"]]
lab = {c: i for i, c in enumerate(top_cls)}; other = len(top_cls)
t_c = np.array([lab.get(c, other) for c in true_f]); p_c = np.array([lab.get(c, other) for c in pred_f])
CM_COMPACT = np.zeros((other + 1, other + 1), np.int32); np.add.at(CM_COMPACT, (t_c, p_c), 1)
CM_LABELS = [VOC.names[c] for c in top_cls] + ["Other"]
print(f"confusion ({BEST_MODEL}): Top-1 closed = {(pred_f == true_f).mean():.3f} | compact {CM_COMPACT.shape} + raw {CM_RAW.shape} ذخیره شد")

# ---------- Viterbi (diagnostic گذشته‌نگر، نه forecasting) ----------
HMM_FINAL = FIT_HMM[BEST["HMM"]["lr_h"]]["params"]
vit_rows, VIT_EX = [], None
for c in np.unique(CAMP_IDX[FINAL_IDX]):
    v = VOC.map[CHAIN[c]]; v = v[v >= 0]
    if len(v) < 2:
        continue
    path = viterbi_np(HMM_FINAL, v); true = VOC.tact[v]
    vit_rows.append(dict(campaign=camp_list[c], n=len(v), acc=float((path == true).mean())))
    if VIT_EX is None or len(v) > len(VIT_EX[1]):
        VIT_EX = (camp_list[c], true, path)
VIT_DF = pd.DataFrame(vit_rows); VIT_DF.to_csv(OUT2 / "viterbi_diagnostic.csv", index=False)
print(f"Viterbi (retrospective): دقت رمزگشایی تاکتیک روی {len(VIT_DF)} زنجیرهٔ FINAL = {VIT_DF['acc'].mean():.3f} (میانگین)  | نمونه: {VIT_EX[0]}")

## Checkpoint: فقط بهترین مدل + کنترل پارامترها + reload
فقط یک فایل `best_model.ckpt` (بهترین مدل بر اساس CV) نگه داشته می‌شود. پس از ذخیره: reload، مقایسهٔ خروجی روی یک batch، بررسی پارامترها (متناهی بودن، stochastic بودن سطرهای HMM، تعداد پارامتر) و مقایسهٔ SHA-256.

In [ ]:
# ============================== Checkpoint + Reload Integrity ==============================
def build_payload(name):
    b = BEST[name]
    pl = dict(model=name, best_cfg=b, cfg=CFG, window=L, vocab_gids=VOC.gids, vocab_names=VOC.names, tact_tok=VOC.tact_tok)
    if name == "HMM" or b["alpha"] < 1.0:
        pl["hmm"] = FIT_HMM[b["lr_h"]]["params"].arrays()
    if name == "VOHMM":
        pl["vom"] = dict(uni=FIT_VOM.uni, tab=FIT_VOM.tab, K=FIT_VOM.K, eps=FIT_VOM.eps)
    if name == "SHHMM":
        pl["shhmm"] = {k: v.detach().cpu() for k, v in FIT_SH["model"].state_dict().items()}
    return pl

def fingerprint(pl):
    h = hashlib.sha256()
    for k in sorted(pl.get("hmm", {})):
        h.update(np.ascontiguousarray(pl["hmm"][k]).tobytes())
    for k in sorted(pl.get("shhmm", {})):
        h.update(pl["shhmm"][k].numpy().tobytes())
    if "vom" in pl:
        v = VOM(len(pl["vom"]["uni"]), pl["vom"]["K"], pl["vom"]["eps"]); v.uni, v.tab = pl["vom"]["uni"], pl["vom"]["tab"]
        h.update(v.fingerprint().encode())
    return h.hexdigest()

def count_params(pl):
    n = sum(a.size for a in pl.get("hmm", {}).values()) + sum(t.numel() for t in pl.get("shhmm", {}).values())
    if "vom" in pl:
        n += sum(len(t) for t in pl["vom"]["tab"][1:])
    return int(n)

def load_predictor(path):
    if HAS_TORCH:
        pl = torch.load(path, map_location="cpu", weights_only=False)
    else:
        with open(path, "rb") as fh:
            pl = pickle.load(fh)
    name, b = pl["model"], pl["best_cfg"]
    hmm_p = HMMParams(**pl["hmm"]) if "hmm" in pl else None
    if name == "HMM":
        fn = lambda tok, ln, tr: hmm_predict(hmm_p, tok, ln, tr)
    elif name == "VOHMM":
        v = VOM(len(pl["vom"]["uni"]), pl["vom"]["K"], pl["vom"]["eps"]); v.uni, v.tab = pl["vom"]["uni"], pl["vom"]["tab"]
        fn = lambda tok, ln, tr: v.predict(tok, ln, b["D"]) if b["alpha"] >= 1 else blend(v.predict(tok, ln, b["D"]), hmm_predict(hmm_p, tok, ln, tr), b["alpha"])
    else:
        n_cls = len(pl["vocab_gids"])
        dummy = hmm_p if hmm_p is not None else MLE_DEV
        net = SHHMM(n_cls, pl["tact_tok"], dummy, CFG["SH_EMB"], CFG["SH_HID"], CFG["SH_LAYERS"], CFG["SH_DROP"]).to(DEVICE)
        net.load_state_dict(pl["shhmm"]); net.eval()
        fn = lambda tok, ln, tr: (shhmm_predict_torch(net, tok, ln, tr) if b["alpha"] >= 1
                                  else blend(shhmm_predict_torch(net, tok, ln, tr), hmm_predict(hmm_p, tok, ln, tr), b["alpha"]))
    return pl, fn

CKPT = OUT2 / "best_model.ckpt"
for p in OUT2.glob("best_model*"):
    p.unlink()                                                    # فقط یک checkpoint نگه داشته می‌شود
PAYLOAD = build_payload(BEST_MODEL); FP_SAVED = fingerprint(PAYLOAD); NP_SAVED = count_params(PAYLOAD)
if HAS_TORCH:
    torch.save(PAYLOAD, CKPT)
else:
    with open(CKPT, "wb") as fh:
        pickle.dump(PAYLOAD, fh)

pl2, fn2 = load_predictor(CKPT)
probe = slice(0, min(256, len(FINAL_IDX)))
P_mem = predict_model(BEST_MODEL, tok_f[probe], ln_f[probe], trc_f[probe]); P_re = fn2(tok_f[probe], ln_f[probe], trc_f[probe])
max_diff = float(np.abs(P_mem - P_re).max())
checks = dict(
    best_model=BEST_MODEL, checkpoint_files=[p.name for p in OUT2.glob("best_model*")], size_mb=round(CKPT.stat().st_size / 1e6, 3),
    sha256_match=(fingerprint(pl2) == FP_SAVED), n_params_match=(count_params(pl2) == NP_SAVED), n_params=NP_SAVED,
    max_abs_prob_diff_after_reload=max_diff,
    all_finite=bool(np.isfinite(P_re).all()), probs_sum_to_one=bool(np.allclose(P_re.sum(1), 1, atol=1e-6)),
    hmm_stochastic=(HMMParams(**pl2["hmm"]).check() if "hmm" in pl2 else None),
    vocab_identical=bool(np.array_equal(pl2["vocab_gids"], VOC.gids)))
assert checks["sha256_match"] and checks["n_params_match"] and checks["all_finite"] and checks["probs_sum_to_one"]
assert max_diff < 1e-5, f"اختلاف خروجی بعد از reload: {max_diff}"
assert len(checks["checkpoint_files"]) == 1
json.dump(checks, open(OUT2 / "checkpoint_report.json", "w"), indent=2)
print(json.dumps(checks, indent=2))

## Case study: Top-5 برای یک Group یا یک prefix دستی
- **prefix دستی:** اگر دقیقاً با ورودی یک نمونهٔ واقعی دیتاست (حالت واقعی یک زنجیره) برابر باشد، target قابل‌مقایسه گزارش می‌شود؛ در غیر این صورت فقط پیش‌بینی گزارش می‌شود و ground-truth ادعا نمی‌شود.
- **Group دلخواه:** `CASE_GROUP` را با نام کمپین/گروه تنظیم کنید (پیش‌فرض: بزرگ‌ترین کمپین FINAL test).

In [ ]:
# ============================== Case study ==============================
INPUT_TO_TARGETS, INPUT_TO_GROUPS = collections.defaultdict(set), collections.defaultdict(set)
for i, s in enumerate(SAMPLES):
    INPUT_TO_TARGETS[tuple(s["input"])].add(s["target"]); INPUT_TO_GROUPS[tuple(s["input"])].add(int(GROUP_IDX[i]))

def topk_for_prefix(prefix, k=5, model=None):
    model = model or BEST_MODEL
    ids = list(prefix); w = ids[-L:]
    tok = np.zeros((1, L), np.int64)
    for j, t in enumerate(w):
        g = GID.get(t, -1); v = VOC.map[g] if g >= 0 else -1
        tok[0, j] = 1 if v < 0 else v + 2
    P = predict_model(model, tok, np.array([len(w)]), np.array([len(ids) > L]))[0]
    top = np.argsort(-P)[:k]
    res = [dict(rank=r + 1, technique=VOC.names[c], name=TECH_NAME.get(VOC.names[c], ""), tactic=TACTIC_ORDER[VOC.tact[c]], prob=float(P[c]))
           for r, c in enumerate(top)]
    key = tuple(prefix); truth = INPUT_TO_TARGETS.get(key)
    info = dict(model=model, prefix_len=len(ids), unknown_in_vocab=[t for t in w if (GID.get(t, -1) < 0 or VOC.map[GID.get(t, 0)] < 0)])
    if truth:
        gs = INPUT_TO_GROUPS[key]
        info.update(ground_truth=sorted(truth), truth_in_top=[r["technique"] for r in res if r["technique"] in truth],
                    from_final_test_only=all(g in set(FINAL_GROUPS) for g in gs),
                    note="prefix با حالت واقعی یک زنجیره در دیتاست برابر است")
    else:
        info.update(ground_truth=None, note="prefix با هیچ حالت واقعی دیتاست برابر نیست؛ فقط پیش‌بینی گزارش می‌شود (ground-truth ادعا نمی‌شود)")
    return res, info

def show_case(title, prefix, k=5):
    res, info = topk_for_prefix(prefix, k)
    print(f"\n=== {title} | مدل: {info['model']} | طول prefix: {info['prefix_len']}")
    print("prefix (آخرین ۶):", prefix[-6:])
    for r in res:
        print(f"  {r['rank']}. {r['technique']:10s} {r['prob']:.3f}  [{r['tactic']}]  {r['name']}")
    print("  ->", info["note"], "| ground-truth:", info["ground_truth"], "| در Top-5:", info.get("truth_in_top"))
    return res, info

# (الف) prefix دستی (مثال شما)
CASE_PREFIX = ["T1588.002", "T1189", "T1566.001", "T1053.002", "T1053.005", "T1059.001"]
CASE_RES = {"manual": show_case("prefix دستی", CASE_PREFIX)}

# (ب) یک Group دلخواه؛ پیش‌فرض: بزرگ‌ترین کمپین FINAL test
CASE_GROUP = None
fc = CAMP_IDX[FINAL_IDX]
cname = CASE_GROUP or camp_list[np.bincount(fc).argmax()]
cs = [i for i in np.flatnonzero(camp_list[CAMP_IDX] == cname)]
assert cs, "نام کمپین پیدا نشد"
mid = sorted(cs, key=lambda i: ORIG_LEN[i])[len(cs) // 2]
CASE_RES["group"] = show_case(f"Group = {cname}", SAMPLES[mid]["input"])

## شکل‌های آمادهٔ انتشار (IEEE) — PDF برداری + PNG با ۶۰۰ dpi
عرض یک ستون ۳٫۵ in و دو ستون ۷٫۱۶ in، فونت serif، رنگ‌های سازگار با کوررنگی. خروجی در پوشهٔ `figures/`.

In [ ]:
# ============================== سبک IEEE و توابع کمکی ==============================
warnings.filterwarnings("ignore", message=".*non-interactive.*")
W1, W2 = 3.5, 7.16
COL = {"HMM": "#0072B2", "SHHMM": "#D55E00", "VOHMM": "#009E73"}
plt.rcParams.update({
    "font.family": "serif", "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"], "mathtext.fontset": "dejavuserif",
    "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8, "legend.fontsize": 6.5, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "axes.linewidth": 0.6, "lines.linewidth": 1.0, "axes.grid": True, "grid.alpha": 0.3, "grid.linewidth": 0.4,
    "pdf.fonttype": 42, "ps.fonttype": 42, "figure.dpi": 110, "savefig.dpi": 600, "legend.frameon": False})
FIG_FILES = []
def save_fig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight"); fig.savefig(FIG_DIR / f"{name}.png", dpi=600, bbox_inches="tight")
    FIG_FILES.append(name); plt.show(); plt.close(fig)

# --- Fig 1: طول زنجیرهٔ گروه‌ها ---
chain_len = np.array([len(c) for c in CHAIN])
fig, ax = plt.subplots(figsize=(W1, 2.3))
ax.hist(chain_len, bins=np.logspace(0, np.log10(chain_len.max() + 1), 25), color="#4C72B0", edgecolor="white", linewidth=0.3)
ax.set_xscale("log")
for v, ls, lb in ((np.mean(chain_len), "--", f"mean={np.mean(chain_len):.1f}"), (np.median(chain_len), ":", f"median={np.median(chain_len):.0f}"), (L, "-", f"window L={L}")):
    ax.axvline(v, color="k", ls=ls, lw=0.8, label=lb)
ax.set_xlabel("Distinct techniques per campaign (log scale)"); ax.set_ylabel("Campaigns"); ax.legend()
save_fig(fig, "fig01_group_sequence_length")

# --- Fig 2: پوشش تاکتیک ---
cov_c = np.zeros(NT, int)
for c in CHAIN:
    cov_c[np.unique(TACT_OF_GID[c])] += 1
tgt_c = np.bincount(GYT, minlength=NT)
fig, axs = plt.subplots(1, 2, figsize=(W2, 2.6), sharey=True)
y_ = np.arange(NT)
axs[0].barh(y_, cov_c, color="#4C72B0"); axs[0].set_xlabel("Campaigns using the tactic"); axs[0].set_yticks(y_); axs[0].set_yticklabels(TACTIC_ORDER); axs[0].invert_yaxis()
axs[1].barh(y_, tgt_c, color="#DD8452"); axs[1].set_xlabel("Samples with this target tactic")
fig.tight_layout(); save_fig(fig, "fig02_tactic_coverage")

# --- Fig 3: توزیع پنجرهٔ لغزان ---
fig, ax = plt.subplots(figsize=(W1, 2.3))
bins = np.arange(0, 80, 2)
ax.hist(np.minimum(ORIG_LEN, 79), bins=bins, alpha=0.55, color="#999999", label="Full history (clipped at 79)")
ax.hist(GLEN, bins=bins, alpha=0.85, color="#0072B2", label=f"Sliding window L={L}")
ax.set_yscale("log"); ax.set_xlabel("Input length (techniques)"); ax.set_ylabel("Samples")
ax.set_title(f"{GTRUNC.mean():.1%} of samples truncated by the window"); ax.legend()
save_fig(fig, "fig03_sliding_window_distribution")

# --- Fig 4: مقایسهٔ CV ---
fig, axs = plt.subplots(1, 2, figsize=(W2, 2.4))
for ax, met, ttl in ((axs[0], "mrr_open", "MRR"), (axs[1], "top5_open", "Top-5")):
    data = [cv_df.loc[cfg_mask(cv_df, BEST_KEY[m]), met].values for m in MODELS]
    bp = ax.boxplot(data, labels=MODELS, patch_artist=True, widths=0.5, medianprops=dict(color="k"))
    for p, m in zip(bp["boxes"], MODELS):
        p.set_facecolor(COL[m]); p.set_alpha(0.6)
    for i, d in enumerate(data):
        ax.scatter(np.random.default_rng(i).normal(i + 1, 0.04, len(d)), d, s=6, color="k", zorder=3)
    ax.set_ylabel(f"{ttl} (open vocabulary)"); ax.set_title(f"{CFG['N_FOLDS']}-fold GroupKFold")
fig.tight_layout(); save_fig(fig, "fig04_cv_model_comparison")

# --- Fig 5: Top-k نهایی ---
fig, ax = plt.subplots(figsize=(W1 + 0.6, 2.4)); wd = 0.8 / len(MODELS)
for j, m in enumerate(MODELS):
    r = CI_FINAL[CI_FINAL.model == m].set_index("metric")
    pts = [r.loc[k, "point"] for k in ("top1", "top3", "top5")]
    err = [[r.loc[k, "point"] - r.loc[k, "lo"] for k in ("top1", "top3", "top5")], [r.loc[k, "hi"] - r.loc[k, "point"] for k in ("top1", "top3", "top5")]]
    ax.bar(np.arange(3) + j * wd - 0.4 + wd / 2, pts, wd, yerr=err, capsize=2, color=COL[m], label=m, error_kw=dict(lw=0.6))
ax.set_xticks(range(3)); ax.set_xticklabels(["Top-1", "Top-3", "Top-5"]); ax.set_ylabel("Accuracy (FINAL test)"); ax.legend()
save_fig(fig, "fig05_final_topk")

# --- Fig 6: ROC-AUC (ECDF AUC هر کلاس) ---
fig, ax = plt.subplots(figsize=(W1 + 0.6, 2.5))
for m in MODELS:
    per_cls = FINAL_RES[m]["auc"][0]
    if per_cls is not None and len(per_cls):
        xs = np.sort(per_cls); ax.step(xs, np.arange(1, len(xs) + 1) / len(xs), where="post", color=COL[m],
                                       label=f"{m} (macro AUC={FINAL_RES[m]['metrics']['auc_macro']:.3f})")
ax.set_xlabel("One-vs-rest ROC-AUC per technique (closed vocabulary)"); ax.set_ylabel("ECDF"); ax.legend(loc="upper left")
save_fig(fig, "fig06_roc_auc")

# --- Fig 7: تکنیک‌های نادر ---
fig, ax = plt.subplots(figsize=(W1 + 0.6, 2.4)); wd = 0.8 / len(MODELS)
for j, m in enumerate(MODELS):
    t = RARE_OOF[m]; ax.bar(np.arange(len(t)) + j * wd - 0.4 + wd / 2, t["top5"].fillna(0), wd, color=COL[m], label=m)
ax.set_xticks(range(len(RARE_BINS))); ax.set_xticklabels([f"{b[2]}\n(n={n})" for b, n in zip(RARE_BINS, RARE_OOF[MODELS[0]]["n"])])
ax.set_xlabel("Target frequency in training folds"); ax.set_ylabel("Top-5 (out-of-fold)"); ax.legend()
save_fig(fig, "fig07_rare_technique_analysis")

# --- Fig 8: طول prefix ---
fig, ax = plt.subplots(figsize=(W1 + 0.6, 2.4))
for m in MODELS:
    t = PREFIX_OOF[m]; ax.plot(range(len(t)), t["mrr"], marker="o", ms=3, color=COL[m], label=m)
ax.set_xticks(range(len(PREFIX_BINS))); ax.set_xticklabels([b[2] for b in PREFIX_BINS])
ax.set_xlabel("Prefix length (techniques observed)"); ax.set_ylabel("MRR (out-of-fold)"); ax.legend()
save_fig(fig, "fig08_prefix_length_analysis")

# --- Fig 9: Confusion matrix ---
cmn = CM_COMPACT / np.maximum(CM_COMPACT.sum(1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(W2 * 0.75, W2 * 0.7))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(CM_LABELS))); ax.set_xticklabels(CM_LABELS, rotation=90, fontsize=5.5)
ax.set_yticks(range(len(CM_LABELS))); ax.set_yticklabels(CM_LABELS, fontsize=5.5); ax.grid(False)
ax.set_xlabel("Predicted (top-1)"); ax.set_ylabel("True target"); ax.set_title(f"{BEST_MODEL}: top-{CFG['CM_TOP']} techniques + Other (row-normalised)")
fig.colorbar(im, fraction=0.04, pad=0.02); fig.tight_layout(); save_fig(fig, "fig09_confusion_matrix")

# --- Fig 10: Viterbi diagnostic ---
fig, axs = plt.subplots(1, 2, figsize=(W2, 2.4))
axs[0].hist(VIT_DF["acc"], bins=np.linspace(0, 1, 11), color="#4C72B0", edgecolor="white", linewidth=0.3)
axs[0].set_xlabel("Tactic-decoding accuracy per campaign"); axs[0].set_ylabel("Campaigns (FINAL)")
nm, tr_, pa_ = VIT_EX
axs[1].step(range(len(tr_)), tr_, where="mid", color="k", label="True tactic"); axs[1].step(range(len(pa_)), pa_ + 0.12, where="mid", color="#D55E00", ls="--", label="Viterbi")
axs[1].set_xlabel("Position in chain"); axs[1].set_ylabel("Tactic rank"); axs[1].set_title(f"{nm} (retrospective)"); axs[1].legend()
fig.tight_layout(); save_fig(fig, "fig10_viterbi_diagnostic")

# --- Fig 11: Case study Top-5 ---
fig, axs = plt.subplots(1, 2, figsize=(W2, 2.4))
for ax, (key, ttl) in zip(axs, (("manual", "Manual prefix"), ("group", f"Group: {cname[:22]}"))):
    res, info = CASE_RES[key]; names = [r["technique"] for r in res][::-1]; pr = [r["prob"] for r in res][::-1]
    truth = set(info.get("ground_truth") or [])
    ax.barh(names, pr, color=["#009E73" if n in truth else "#999999" for n in names])
    ax.set_xlabel("Predicted probability"); ax.set_title(f"{ttl} ({info['model']})")
fig.tight_layout(); save_fig(fig, "fig11_case_study_top5")

# --- Fig 12: train loss (بدون validation) ---
hists = {"HMM": FIT_HMM[BEST["HMM"]["lr_h"]]["hist"], "SHHMM": (FIT_SH["hist"] if FIT_SH else [])}
if any(len(h) for h in hists.values()):
    fig, ax = plt.subplots(figsize=(W1 + 0.6, 2.4))
    for m, h in hists.items():
        if h:
            ax.plot([r["epoch"] for r in h], [r["train_nll"] for r in h], color=COL[m], label=f"{m} train NLL")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Train NLL (final fit on Development)"); ax.legend()
    save_fig(fig, "fig12_train_loss")
else:
    print("منحنی train loss: در حالت MLE (بدون PyTorch) آموزش گرادیانی وجود ندارد؛ روی ماشین GPU تولید می‌شود.")
print("figures saved:", FIG_FILES, "->", FIG_DIR)

In [ ]:
# ============================== خلاصهٔ نهایی و ذخیرهٔ تنظیمات ==============================
SUMMARY = dict(
    n_samples=N_S, n_groups=int(len(GROUP_NAMES)), window=L, n_folds=CFG["N_FOLDS"],
    final_groups=int(len(FINAL_GROUPS)), final_samples=int(len(FINAL_IDX)), dev_samples=int(len(DEV_IDX)),
    best_model=BEST_MODEL, best_cfg={m: {k: (None if v == NA else v) for k, v in BEST[m].items()} for m in MODELS},
    torch=bool(HAS_TORCH), device=str(DEVICE), final_metrics={m: {k: float(v) for k, v in FINAL_RES[m]["metrics"].items()} for m in MODELS})
json.dump(SUMMARY, open(OUT2 / "results_summary.json", "w"), indent=2, default=str)
json.dump(CFG, open(OUT2 / "config.json", "w"), indent=2)
print(FINAL_TABLE[["top1_open", "top3_open", "top5_open", "mrr_open", "gmrr_open", "auc_macro"]].round(4).to_string())
print("\nفایل‌های خروجی:"); [print("  ", p.relative_to(OUT2), f"{p.stat().st_size/1e3:.0f} KB") for p in sorted(OUT2.rglob("*")) if p.is_file() and p.suffix != ".png"]